In [ ]:
pip install tensorflow opencv-python

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install fvcore


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=d76310786a7bff0cf8dbbce063336ad2d14fc375d63cfc546148623b4ff681b1
  Stored in directory: /root/.cache/pip/wheels/65/71/95/3b8fde5c65c6e4a806e0867c1651dcc71a1cb2f3430e8f355f
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31527 sha256=06b44c846d3171db604fcdaad739625224c08ebf0c33036fcb5e8a21f61205aa
  Stored in directory: /root/.cache/pip/wheels/ba/5e/16/6117f8fe7e9c0c161a795e10d94645ebcf301ccbd01f66d8ec
Successfully built fvcore iopath


# CORRECT ENSEMBLE KD with Transfer Learning

In [ ]:
#!/usr/bin/env python3
# === Strict TL + KD + Pruning (filenames unchanged) ===
# - Generic TL: full fine-tuning on feline, then full fine-tuning on human
# - No head-only phases (removes odd behavior & risk of leakage via mis-freezing)
# - Strong leakage guards: fixed splits, no aug on val/test, strict eval()
# - Regularization: label smoothing, weight decay, grad clipping, early stopping
# - Pruning: global L1 prune on Conv/Linear, brief recovery finetune
# - Filenames & SAVE_ROOT kept EXACTLY as your original (so quantization snippets still work)
#
# Prints: each teacher's test metrics, ensemble test metrics, and KD student test metrics

import os, time, json, math, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

# -------- Optional deps (kept) --------
try:
    from fvcore.nn import FlopCountAnalysis
    FVCORE_OK = True
except Exception:
    FVCORE_OK = False

try:
    import onnx
    import onnxruntime as ort
    from onnxruntime.quantization import quantize_dynamic, QuantType
    ORT_OK = True
except Exception:
    ORT_OK = False

# ====================== CONFIG ======================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CPU = torch.device("cpu")

FELINE_IMG_PATH = "/kaggle/input/cat-reticulocyte/felineAlldata"
HUMAN_IMG_PATH  = "/kaggle/input/human-reticulocyte/SubsetAlldata"

SAVE_ROOT = Path("outputs") / "kd_transfer_onnx_table9"  # <== unchanged
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

# training schedule (human student epochs kept as-is)
EPOCHS_FELINE_FULL   = 6   # feline full FT (replaces frozen+unfrozen)
EPOCHS_HUMAN_FULL    = 18  # human full FT (replaces frozen+unfrozen)
EPOCHS_STUDENT       = 20

BATCH_SIZE = 32
LR_FELINE_FULL   = 5e-5
LR_HUMAN_FULL    = 5e-5
WEIGHT_DECAY     = 1e-4
ALPHA            = 0.5
TEMPERATURE      = 4.0
GRAD_CLIP_NORM   = 2.0

NUM_WORKERS = 2
IMG_SIZE    = 224
VAL_FRAC    = 0.2
TEST_FRAC   = 0.2
FELINE_VAL_FRAC = 0.2

# Benchmark settings (kept)
FAST_BENCH = True
if FAST_BENCH:
    WARMUP_ITERS, BENCH_ITERS, BENCH_BATCH = 2, 5, 8
else:
    WARMUP_ITERS, BENCH_ITERS, BENCH_BATCH = 10, 40, 32

# ONNX/ORT providers (kept)
ORT_PROVIDERS = ["CPUExecutionProvider"]

# ====================== DATA ======================
mean_imnet = [0.485, 0.456, 0.406]
std_imnet  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])

# Build datasets without leaking transforms into the split logic
feline_base = datasets.ImageFolder(FELINE_IMG_PATH, transform=None)
human_base  = datasets.ImageFolder(HUMAN_IMG_PATH, transform=None)
assert len(feline_base.classes) >= 2, "Feline dataset must have >=2 classes."
NUM_CLASSES = len(human_base.classes)

def stratified_split_indices_by_frac(base_ds, val_frac, test_frac=0.0, seed=SEED):
    """Split on base_ds.samples (paths+targets) to avoid transform side-effects; stratified & seeded."""
    rng = np.random.default_rng(seed)
    targets = np.array([y for _, y in base_ds.samples])
    idx = np.arange(len(targets))
    train_idx, val_idx, test_idx = [], [], []
    for c in np.unique(targets):
        ci = idx[targets==c].copy(); rng.shuffle(ci)
        n = len(ci)
        n_val  = max(1, int(round(n*val_frac)))
        n_test = max(0, int(round(n*test_frac)))
        n_val = min(n_val, n-1) if n>1 else 1
        val_idx.extend(ci[:n_val])
        test_idx.extend(ci[n_val:n_val+n_test])
        train_idx.extend(ci[n_val+n_test:])
    return train_idx, val_idx, test_idx

# feline split
f_tr_idx, f_va_idx, _ = stratified_split_indices_by_frac(feline_base, FELINE_VAL_FRAC, 0.0)
feline_train = Subset(datasets.ImageFolder(FELINE_IMG_PATH, transform=train_tf), f_tr_idx)
feline_val   = Subset(datasets.ImageFolder(FELINE_IMG_PATH, transform=eval_tf),  f_va_idx)

# human split
h_tr_idx, h_va_idx, h_te_idx = stratified_split_indices_by_frac(human_base, VAL_FRAC, TEST_FRAC)
human_train = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=train_tf), h_tr_idx)
human_val   = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  h_va_idx)
human_test  = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  h_te_idx)

print("Feline:", len(feline_train), "train,", len(feline_val), "val", datasets.ImageFolder(FELINE_IMG_PATH).classes)
print("Human :", len(human_train), "train,", len(human_val), "val,", len(human_test), "test", datasets.ImageFolder(HUMAN_IMG_PATH).classes)

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True)

feline_train_loader = make_loader(feline_train, shuffle=True)
feline_val_loader   = make_loader(feline_val,   shuffle=False)
human_train_loader  = make_loader(human_train,  shuffle=True)
human_val_loader    = make_loader(human_val,    shuffle=False)
human_test_loader   = make_loader(human_test,   shuffle=False)

# ====================== UTILS ======================
def replace_classifier_for_num_classes(model, num_classes):
    if isinstance(model, models.SqueezeNet):
        model.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1,1))
        model.num_classes = num_classes
    elif hasattr(model, 'classifier') and isinstance(model.classifier, nn.Sequential):
        in_features = None
        for layer in reversed(model.classifier):
            if isinstance(layer, nn.Linear):
                in_features = layer.in_features; break
        if in_features is None and hasattr(model, 'fc'):
            in_features = model.fc.in_features
            model.fc = nn.Linear(in_features, num_classes)
        else:
            model.classifier[-1] = nn.Linear(in_features, num_classes)
    elif hasattr(model, 'fc'):
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
    else:
        raise ValueError("Unsupported model type.")
    return model

def count_params_m(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def compute_flops_g(model, input_size=(1,3,IMG_SIZE,IMG_SIZE)):
    if not FVCORE_OK: return float("nan")
    m = model.to(CPU).eval()
    x = torch.randn(*input_size)
    try:
        return float(FlopCountAnalysis(m, x).total() / 1e9)
    except Exception:
        return float("nan")

def save_state(model, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), path)
    return path.stat().st_size / (1024*1024)

def optim_for(params, lr):
    return torch.optim.Adam(params, lr=lr, weight_decay=WEIGHT_DECAY)

def run_epoch(model, loader, loss_fn, opt=None, device=DEVICE, grad_clip=None):
    train = opt is not None
    model.train(train)
    total, correct, n = 0.0, 0, 0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        if train: opt.zero_grad(set_to_none=True)
        out = model(x)
        loss = loss_fn(out, y)
        if train:
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
        total += loss.item()*x.size(0)
        correct += (out.argmax(1)==y).sum().item()
        n += x.size(0)
    return total/max(n,1), correct/max(n,1)

@torch.no_grad()
def eval_metrics(model, loader, device=DEVICE):
    model.eval()
    y_true, y_pred = [], []
    for x,y in loader:
        x = x.to(device)
        y_true.extend(y.numpy().tolist())
        y_pred.extend(model(x).argmax(1).cpu().numpy().tolist())
    acc = accuracy_score(y_true, y_pred)*100.0
    f1m = f1_score(y_true, y_pred, average="macro")*100.0
    return acc, f1m

class EarlyStopper:
    def __init__(self, patience=5, mode="max", delta=0.0):
        self.patience, self.mode, self.delta = patience, mode, delta
        self.best = -float("inf") if mode=="max" else float("inf")
        self.count = 0
    def step(self, metric):
        improve = (metric > self.best + self.delta) if self.mode=="max" else (metric < self.best - self.delta)
        if improve:
            self.best = metric; self.count = 0; return True
        else:
            self.count += 1; return False
    def should_stop(self): return self.count >= self.patience

# ====================== Pruning helpers ======================
import torch.nn.utils.prune as prune

def modules_to_prune(model):
    pairs = []
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            pairs.append((m, 'weight'))
    return pairs

def apply_global_prune(model, amount=0.2):
    params = modules_to_prune(model)
    if len(params)==0: return
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=amount)
    # Convert pruned weights into real tensors (remove masks & reparam)
    for m, name in params:
        try:
            prune.remove(m, name)
        except Exception:
            pass

# ====================== Generic TL Teacher ======================
def pretrain_teacher(arch_name, model_fn):
    out_dir = SAVE_ROOT / f"teacher_{arch_name}"
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) Feline full fine-tune
    model = model_fn(weights="DEFAULT" if "weights" in model_fn.__code__.co_varnames else None)
    model = replace_classifier_for_num_classes(model, len(datasets.ImageFolder(FELINE_IMG_PATH).classes)).to(DEVICE)
    ce_feline = nn.CrossEntropyLoss(label_smoothing=0.1)
    opt = optim_for(model.parameters(), LR_FELINE_FULL)
    es_feline = EarlyStopper(patience=3, mode="max")
    best_feline_path = out_dir / f"{arch_name}_feline_best.pth"  # <== unchanged filename
    best_feline_acc = -1.0

    for ep in range(1, EPOCHS_FELINE_FULL+1):
        tr_loss, tr_acc = run_epoch(model, feline_train_loader, ce_feline, opt, DEVICE, grad_clip=GRAD_CLIP_NORM)
        v_acc, v_f1 = eval_metrics(model, feline_val_loader, DEVICE)
        print(f"[{arch_name}][Feline] {ep}/{EPOCHS_FELINE_FULL} loss={tr_loss:.6f} val_acc={v_acc:.2f} f1={v_f1:.2f}")
        if es_feline.step(v_acc):
            save_state(model, best_feline_path); best_feline_acc = v_acc
        if es_feline.should_stop(): break

    model.load_state_dict(torch.load(best_feline_path, map_location=DEVICE))

    # 2) Switch head for human & full fine-tune
    model = replace_classifier_for_num_classes(model, NUM_CLASSES).to(DEVICE)
    ce_human = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt = optim_for(model.parameters(), LR_HUMAN_FULL)
    es_human = EarlyStopper(patience=4, mode="max")
    best_human_path = out_dir / f"{arch_name}_human_best.pth"  # <== unchanged filename
    best_human_acc = -1.0

    for ep in range(1, EPOCHS_HUMAN_FULL+1):
        tr_loss, tr_acc = run_epoch(model, human_train_loader, ce_human, opt, DEVICE, grad_clip=GRAD_CLIP_NORM)
        v_acc, v_f1 = eval_metrics(model, human_val_loader, DEVICE)
        print(f"[{arch_name}][Human] {ep}/{EPOCHS_HUMAN_FULL} loss={tr_loss:.6f} val_acc={v_acc:.2f} f1={v_f1:.2f}")
        if es_human.step(v_acc):
            save_state(model, best_human_path); best_human_acc = v_acc
        if es_human.should_stop(): break

    model.load_state_dict(torch.load(best_human_path, map_location=DEVICE))

    # 3) Prune + short recovery finetune on human train (strictness against overfit)
    apply_global_prune(model, amount=0.2)  # 20% global L1
    opt = optim_for(model.parameters(), LR_HUMAN_FULL/2)
    es_rec = EarlyStopper(patience=3, mode="max")
    for ep in range(1, 6):  # brief recovery
        tr_loss, tr_acc = run_epoch(model, human_train_loader, ce_human, opt, DEVICE, grad_clip=GRAD_CLIP_NORM)
        v_acc, v_f1 = eval_metrics(model, human_val_loader, DEVICE)
        print(f"[{arch_name}][Prune-Recover] {ep}/5 loss={tr_loss:.6f} val_acc={v_acc:.2f} f1={v_f1:.2f}")
        if es_rec.step(v_acc):
            save_state(model, best_human_path); best_human_acc = v_acc
        if es_rec.should_stop(): break

    model.load_state_dict(torch.load(best_human_path, map_location=DEVICE))
    return model, out_dir

# ====================== Train Teachers ======================
teachers_cfg = [
    ("mobilenet_v2",    models.mobilenet_v2),
    ("efficientnet_b0", models.efficientnet_b0),
    ("squeezenet1_1",   models.squeezenet1_1),
]
teachers = {}
for name, fn in teachers_cfg:
    print(f"\n==== Train Teacher: {name} ====")
    model, out_dir = pretrain_teacher(name, fn)
    acc, f1m = eval_metrics(model, human_test_loader, DEVICE)
    teachers[name] = {"model": model, "dir": out_dir, "test_acc": acc, "test_f1m": f1m}
    print(f"[{name}] TEST  acc={acc:.2f}  f1={f1m:.2f}")

# ====================== Ensemble Eval (soft voting) ======================
@torch.no_grad()
def ensemble_eval(teachers, loader, device=DEVICE):
    models_list = [t["model"].to(device).eval() for t in teachers.values()]
    y_true, y_pred = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        probs = None
        for m in models_list:
            p = F.softmax(m(imgs), dim=1)
            probs = p if probs is None else probs + p
        preds = (probs/len(models_list)).argmax(1).cpu().numpy().tolist()
        y_pred.extend(preds); y_true.extend(labels.numpy().tolist())
    acc = accuracy_score(y_true, y_pred)*100.0
    f1m = f1_score(y_true, y_pred, average="macro")*100.0
    return acc, f1m

ens_acc, ens_f1m = ensemble_eval(teachers, human_test_loader, DEVICE)
print(f"\n[Ensemble] TEST acc={ens_acc:.2f}  f1={ens_f1m:.2f}")

# ====================== KD Student ======================
class StudentNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = models.shufflenet_v2_x0_5(weights="DEFAULT")
        in_f = base.fc.in_features
        # add mild dropout for regularization
        base.fc = nn.Sequential(nn.Dropout(p=0.2), nn.Linear(in_f, num_classes))
        self.model = base
    def forward(self, x): return self.model(x)

student = StudentNet(NUM_CLASSES).to(DEVICE)
opt_s = torch.optim.Adam(student.parameters(), lr=1e-4, weight_decay=WEIGHT_DECAY)
ce_s  = nn.CrossEntropyLoss(label_smoothing=0.05)
kl = nn.KLDivLoss(reduction="batchmean")
t_models = [t["model"].to(DEVICE).eval() for t in teachers.values()]

def kd_epoch(student, loader):
    student.train()
    total, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            ps = [F.softmax(m(imgs)/TEMPERATURE, dim=1) for m in t_models]
            t_soft = sum(ps)/len(ps)
        logits = student(imgs)
        loss = ALPHA*ce_s(logits, labels) + (1-ALPHA)*kl(F.log_softmax(logits/TEMPERATURE, dim=1), t_soft)
        opt_s.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP_NORM)
        opt_s.step()
        total += loss.item()*imgs.size(0)
        correct += (logits.argmax(1)==labels).sum().item(); n += imgs.size(0)
    return total/max(n,1), correct/max(n,1)

best_v, best_path = -1.0, SAVE_ROOT / "student_best.pth"  # <== unchanged filename
es_stud = EarlyStopper(patience=5, mode="max")

for ep in range(1, EPOCHS_STUDENT+1):
    tr_loss, tr_acc = kd_epoch(student, human_train_loader)
    v_acc, v_f1 = eval_metrics(student, human_val_loader, DEVICE)
    print(f"[Student KD] {ep}/{EPOCHS_STUDENT} loss={tr_loss:.6f} train_acc={tr_acc*100:.2f} val_acc={v_acc:.2f} f1={v_f1:.2f}")
    if es_stud.step(v_acc):
        save_state(student, best_path); best_v = v_acc
    if es_stud.should_stop(): break

student.load_state_dict(torch.load(best_path, map_location=DEVICE))

# Optional: light prune student & recover briefly (tighten generalization), then re-save SAME filename
apply_global_prune(student, amount=0.15)
opt_rec = torch.optim.Adam(student.parameters(), lr=7.5e-5, weight_decay=WEIGHT_DECAY)
es_re = EarlyStopper(patience=3, mode="max")
for ep in range(1, 5):
    tr_loss, tr_acc = kd_epoch(student, human_train_loader)
    v_acc, v_f1 = eval_metrics(student, human_val_loader, DEVICE)
    print(f"[Student Prune-Recover] {ep}/4 loss={tr_loss:.6f} val_acc={v_acc:.2f} f1={v_f1:.2f}")
    if es_re.step(v_acc): save_state(student, best_path)
    if es_re.should_stop(): break
student.load_state_dict(torch.load(best_path, map_location=DEVICE))

# Final strict evaluations
for name, t in teachers.items():
    acc, f1m = eval_metrics(t["model"], human_test_loader, DEVICE)
    print(f"[{name}] FINAL TEST acc={acc:.2f} f1={f1m:.2f}")
ens_acc, ens_f1m = ensemble_eval(teachers, human_test_loader, DEVICE)
print(f"[Ensemble] FINAL TEST acc={ens_acc:.2f} f1={ens_f1m:.2f}")

stud_acc, stud_f1 = eval_metrics(student, human_test_loader, DEVICE)
print(f"[Student] FINAL TEST acc={stud_acc:.2f} f1={stud_f1:.2f}")


Feline: 2354 train, 588 val ['erythrocyte', 'reticulocyte']
Human : 872 train, 291 val, 291 test ['BG', 'erythrocyte', 'reticulocyte']

==== Train Teacher: mobilenet_v2 ====
[mobilenet_v2][Feline] 1/6 loss=0.660858 val_acc=67.35 f1=40.24
[mobilenet_v2][Feline] 2/6 loss=0.643127 val_acc=67.35 f1=40.24
[mobilenet_v2][Feline] 3/6 loss=0.641209 val_acc=67.35 f1=40.24
[mobilenet_v2][Feline] 4/6 loss=0.634682 val_acc=70.75 f1=52.25
[mobilenet_v2][Feline] 5/6 loss=0.632045 val_acc=71.09 f1=53.44
[mobilenet_v2][Feline] 6/6 loss=0.623056 val_acc=70.41 f1=49.57
[mobilenet_v2][Human] 1/18 loss=1.079108 val_acc=48.45 f1=44.18
[mobilenet_v2][Human] 2/18 loss=1.036894 val_acc=65.29 f1=65.04
[mobilenet_v2][Human] 3/18 loss=0.971017 val_acc=76.63 f1=76.79
[mobilenet_v2][Human] 4/18 loss=0.890965 val_acc=75.60 f1=75.83
[mobilenet_v2][Human] 5/18 loss=0.871084 val_acc=80.41 f1=80.47
[mobilenet_v2][Human] 6/18 loss=0.815309 val_acc=80.41 f1=80.79
[mobilenet_v2][Human] 7/18 loss=0.817969 val_acc=79.38 f1=

Downloading: "https://download.pytorch.org/models/shufflenetv2_x0.5-f707e7126e.pth" to /root/.cache/torch/hub/checkpoints/shufflenetv2_x0.5-f707e7126e.pth



[Ensemble] TEST acc=92.10  f1=92.21


100%|██████████| 5.28M/5.28M [00:00<00:00, 135MB/s]


[Student KD] 1/20 loss=0.566771 train_acc=51.03 val_acc=76.98 f1=75.17
[Student KD] 2/20 loss=0.556920 train_acc=67.89 val_acc=82.47 f1=81.22
[Student KD] 3/20 loss=0.538418 train_acc=79.36 val_acc=87.29 f1=86.87
[Student KD] 4/20 loss=0.504507 train_acc=80.05 val_acc=86.94 f1=86.59
[Student KD] 5/20 loss=0.452437 train_acc=82.68 val_acc=87.63 f1=87.47
[Student KD] 6/20 loss=0.388724 train_acc=84.06 val_acc=89.35 f1=89.16
[Student KD] 7/20 loss=0.333808 train_acc=84.06 val_acc=90.03 f1=89.88
[Student KD] 8/20 loss=0.293184 train_acc=86.47 val_acc=91.75 f1=91.61
[Student KD] 9/20 loss=0.258688 train_acc=88.53 val_acc=94.50 f1=94.42
[Student KD] 10/20 loss=0.245701 train_acc=89.22 val_acc=94.85 f1=94.82
[Student KD] 11/20 loss=0.236691 train_acc=88.53 val_acc=97.59 f1=97.57
[Student KD] 12/20 loss=0.221390 train_acc=89.56 val_acc=97.59 f1=97.58
[Student KD] 13/20 loss=0.214542 train_acc=90.25 val_acc=96.91 f1=96.85
[Student KD] 14/20 loss=0.188830 train_acc=93.81 val_acc=97.59 f1=97.57
[

In [ ]:
!pip install -U onnx onnxruntime onnxruntime-tools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 86.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 5.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-api-core 1.34.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=

In [ ]:
import onnxruntime as ort
print("ORT:", ort.__version__)
print("Providers:", ort.get_available_providers())
# should include: ['CPUExecutionProvider']


ORT: 1.22.1
Providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


In [ ]:
!pip uninstall -y onnxruntime || true
!pip install -U onnx onnxruntime-gpu onnxruntime-tools


Found existing installation: onnxruntime 1.22.1
Uninstalling onnxruntime-1.22.1:
  Successfully uninstalled onnxruntime-1.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.2/283.2 MB 5.9 MB/s eta 0:00:00:00:0100:01


In [ ]:
import onnxruntime as ort
print("ORT:", ort.__version__)
print("Providers:", ort.get_available_providers())
# expect: ['CUDAExecutionProvider', 'CPUExecutionProvider']  (TensorRT may not be available on Kaggle)


ORT: 1.22.0
Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


# New Attempt

In [ ]:
%%writefile onnx_quant_bench.py
# (paste the full script content here)


Overwriting onnx_quant_bench.py


In [ ]:
!ls -lah | grep onnx_quant_bench.py


-rw-r--r-- 1 root root   39 Aug 14 13:49 onnx_quant_bench.py


In [ ]:
!ls -lah


total 20K
drwxr-xr-x 4 root root 4.0K Aug 14 13:35 .
drwxr-xr-x 5 root root 4.0K Aug 14 13:12 ..
-rw-r--r-- 1 root root   39 Aug 14 13:49 onnx_quant_bench.py
drwxr-xr-x 3 root root 4.0K Aug 14 13:12 outputs
drwxr-xr-x 2 root root 4.0K Aug 14 13:12 .virtual_documents


In [ ]:
!ls outputs/kd_transfer_onnx_table9


ls: cannot access 'outputs/kd_transfer_onnx_table9': No such file or directory


# Fixed Quantized Output

In [1]:
!pip uninstall -y onnx onnxruntime onnxruntime-tools
!pip install --no-cache-dir onnx==1.16.1 onnxruntime==1.20.0 onnxruntime-tools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 210.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 170.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 151.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 203.9 MB/s eta 0:00:00


In [ ]:
import onnx, onnxruntime
print("onnx:", onnx.__version__)
print("onnxruntime:", onnxruntime.__version__)

from onnxruntime.quantization import quantize_static, CalibrationDataReader
print("Quantization available ✅")


In [ ]:
import torch, torch.nn as nn
from torchvision import models, datasets, transforms
from pathlib import Path

# === paths (MUST match your training) ===
SAVE_ROOT = Path("outputs") / "kd_transfer_onnx_table9"
CKPT = {
    "mobilenet_v2": SAVE_ROOT/"teacher_mobilenet_v2"/"mobilenet_v2_human_best.pth",
    "efficientnet_b0": SAVE_ROOT/"teacher_efficientnet_b0"/"efficientnet_b0_human_best.pth",
    "squeezenet1_1": SAVE_ROOT/"teacher_squeezenet1_1"/"squeezenet1_1_human_best.pth",
    "student": SAVE_ROOT/"student_best.pth",
}

# how many classes (use the human dataset you trained on)
HUMAN_IMG_PATH = "/kaggle/input/human-reticulocyte/SubsetAlldata"
num_classes = len(datasets.ImageFolder(HUMAN_IMG_PATH).classes)

# build nets exactly like training
def replace_head(m, n):
    if isinstance(m, models.SqueezeNet):
        m.classifier[1] = nn.Conv2d(512, n, kernel_size=1)
    elif hasattr(m, "classifier") and isinstance(m.classifier, nn.Sequential):
        # MobileNet/EfficientNet
        for i in range(len(m.classifier)-1, -1, -1):
            if isinstance(m.classifier[i], nn.Linear):
                in_f = m.classifier[i].in_features
                m.classifier[i] = nn.Linear(in_f, n)
                break
    elif hasattr(m, "fc"):
        in_f = m.fc.in_features
        m.fc = nn.Linear(in_f, n)
    return m

def make_model(tag):
    if tag=="mobilenet_v2":
        m = models.mobilenet_v2(weights=None)
    elif tag=="efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
    elif tag=="squeezenet1_1":
        m = models.squeezenet1_1(weights=None)
    elif tag=="student":
        base = models.shufflenet_v2_x0_5(weights=None)
        in_f = base.fc.in_features
        base.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_f, num_classes))
        class S(nn.Module):
            def __init__(self, b): super().__init__(); self.model=b
            def forward(self,x): return self.model(x)
        m = S(base)
        return m
    return replace_head(m, num_classes)

def export_onnx(tag, ckpt_path, out_path):
    assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"
    m = make_model(tag)
    m.load_state_dict(torch.load(ckpt_path, map_location="cpu"), strict=False)
    m.eval()
    x = torch.randn(1,3,224,224, dtype=torch.float32)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.onnx.export(
        m, x, str(out_path),
        export_params=True, do_constant_folding=True,
        input_names=["input"], output_names=["logits"],
        dynamic_axes=None, opset_version=13
    )
    print(f"[OK] Exported {tag} -> {out_path}")

# export all 4
export_onnx("mobilenet_v2", CKPT["mobilenet_v2"], SAVE_ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_FP32.onnx")
export_onnx("efficientnet_b0", CKPT["efficientnet_b0"], SAVE_ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_FP32.onnx")
export_onnx("squeezenet1_1", CKPT["squeezenet1_1"], SAVE_ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_FP32.onnx")
export_onnx("student", CKPT["student"], SAVE_ROOT/"onnx_student"/"student_FP32.onnx")


[OK] Exported mobilenet_v2 -> outputs/kd_transfer_onnx_table9/onnx_mobilenet_v2/mobilenet_v2_FP32.onnx
[OK] Exported efficientnet_b0 -> outputs/kd_transfer_onnx_table9/onnx_efficientnet_b0/efficientnet_b0_FP32.onnx
[OK] Exported squeezenet1_1 -> outputs/kd_transfer_onnx_table9/onnx_squeezenet1_1/squeezenet1_1_FP32.onnx
[OK] Exported student -> outputs/kd_transfer_onnx_table9/onnx_student/student_FP32.onnx


In [ ]:
import torch, torch.nn as nn
from torchvision import models, datasets
from pathlib import Path

SAVE_ROOT = Path("outputs") / "kd_transfer_onnx_table9"
CKPT = {
    "mobilenet_v2": SAVE_ROOT/"teacher_mobilenet_v2"/"mobilenet_v2_human_best.pth",
    "efficientnet_b0": SAVE_ROOT/"teacher_efficientnet_b0"/"efficientnet_b0_human_best.pth",
    "squeezenet1_1": SAVE_ROOT/"teacher_squeezenet1_1"/"squeezenet1_1_human_best.pth",
    "student": SAVE_ROOT/"student_best.pth",
}

HUMAN_IMG_PATH = "/kaggle/input/human-reticulocyte/SubsetAlldata"
num_classes = len(datasets.ImageFolder(HUMAN_IMG_PATH).classes)

def replace_head(m, n):
    if isinstance(m, models.SqueezeNet):
        m.classifier[1] = nn.Conv2d(512, n, kernel_size=1)
    elif hasattr(m, "classifier") and isinstance(m.classifier, nn.Sequential):
        for i in range(len(m.classifier)-1, -1, -1):
            if isinstance(m.classifier[i], nn.Linear):
                in_f = m.classifier[i].in_features
                m.classifier[i] = nn.Linear(in_f, n); break
    elif hasattr(m, "fc"):
        in_f = m.fc.in_features; m.fc = nn.Linear(in_f, n)
    return m

def make_model(tag):
    if tag=="mobilenet_v2":
        m = models.mobilenet_v2(weights=None)
    elif tag=="efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
    elif tag=="squeezenet1_1":
        m = models.squeezenet1_1(weights=None)
    elif tag=="student":
        base = models.shufflenet_v2_x0_5(weights=None)
        in_f = base.fc.in_features
        base.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_f, num_classes))
        class S(nn.Module):
            def __init__(self, b): super().__init__(); self.model=b
            def forward(self,x): return self.model(x)
        return S(base)
    return replace_head(m, num_classes)

def export_onnx(tag, ckpt_path, out_path):
    m = make_model(tag)
    m.load_state_dict(torch.load(ckpt_path, map_location="cpu"), strict=False)
    m.eval()
    x = torch.randn(1,3,224,224, dtype=torch.float32)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.onnx.export(
        m, x, str(out_path),
        export_params=True, do_constant_folding=True,
        input_names=["input"], output_names=["logits"],
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},  # <-- dynamic batch
        opset_version=13
    )
    print(f"[OK] Exported {tag} -> {out_path}")

export_onnx("mobilenet_v2", CKPT["mobilenet_v2"], SAVE_ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_FP32.onnx")
export_onnx("efficientnet_b0", CKPT["efficientnet_b0"], SAVE_ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_FP32.onnx")
export_onnx("squeezenet1_1",   CKPT["squeezenet1_1"],   SAVE_ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_FP32.onnx")
export_onnx("student",         CKPT["student"],         SAVE_ROOT/"onnx_student"/"student_FP32.onnx")


[OK] Exported mobilenet_v2 -> outputs/kd_transfer_onnx_table9/onnx_mobilenet_v2/mobilenet_v2_FP32.onnx
[OK] Exported efficientnet_b0 -> outputs/kd_transfer_onnx_table9/onnx_efficientnet_b0/efficientnet_b0_FP32.onnx
[OK] Exported squeezenet1_1 -> outputs/kd_transfer_onnx_table9/onnx_squeezenet1_1/squeezenet1_1_FP32.onnx
[OK] Exported student -> outputs/kd_transfer_onnx_table9/onnx_student/student_FP32.onnx


In [ ]:
import numpy as np
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
from torchvision import datasets, transforms

IMG_SIZE=224
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
calib_ds = datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf)

class CalibReader(CalibrationDataReader):
    def __init__(self, ds, n_batches=20, bs=8):
        self.ds=ds; self.n=n_batches; self.bs=bs; self.i=0
        idx = np.random.default_rng(123).permutation(len(ds))[:n_batches*bs]
        self.batches=[]
        for k in range(0,len(idx),bs):
            xs=[ds[j][0].numpy() for j in idx[k:k+bs]]
            self.batches.append({"input": np.stack(xs,0).astype(np.float32)})
    def get_next(self):
        if self.i>=len(self.batches): return None
        b=self.batches[self.i]; self.i+=1; return b

def qdq_int8(fp32_path, int8_path):
    int8_path.parent.mkdir(parents=True, exist_ok=True)
    quantize_static(
        model_input=str(fp32_path),
        model_output=str(int8_path),
        calibration_data_reader=CalibReader(calib_ds, n_batches=20, bs=8),
        quant_format=QuantFormat.QDQ,
        per_channel=True,
        weight_type=QuantType.QInt8,
        activation_type=QuantType.QUInt8,
        reduce_range=False
    )
    print(f"[OK] Quantized -> {int8_path}")

qdq_int8(SAVE_ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_FP32.onnx",     SAVE_ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_INT8.onnx")
qdq_int8(SAVE_ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_FP32.onnx", SAVE_ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_INT8.onnx")
qdq_int8(SAVE_ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_FP32.onnx",   SAVE_ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_INT8.onnx")
qdq_int8(SAVE_ROOT/"onnx_student"/"student_FP32.onnx",               SAVE_ROOT/"onnx_student"/"student_INT8.onnx")


[OK] Quantized -> outputs/kd_transfer_onnx_table9/onnx_mobilenet_v2/mobilenet_v2_INT8.onnx
[OK] Quantized -> outputs/kd_transfer_onnx_table9/onnx_efficientnet_b0/efficientnet_b0_INT8.onnx
[OK] Quantized -> outputs/kd_transfer_onnx_table9/onnx_squeezenet1_1/squeezenet1_1_INT8.onnx
[OK] Quantized -> outputs/kd_transfer_onnx_table9/onnx_student/student_INT8.onnx


In [ ]:
import onnxruntime as ort, numpy as np, time
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader

def mk_sess(p):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    so.intra_op_num_threads = 4; so.inter_op_num_threads = 1
    return ort.InferenceSession(str(p), sess_options=so, providers=["CPUExecutionProvider"])

def acc_f1(sess, loader):
    nm = sess.get_inputs()[0].name
    y_true, y_pred = [], []
    for x,y in loader:
        arr = x.numpy().astype(np.float32)
        logits = sess.run(None, {nm: arr})[0]
        y_pred.extend(np.argmax(logits,1).tolist())
        y_true.extend(y.numpy().tolist())
    return accuracy_score(y_true,y_pred)*100, f1_score(y_true,y_pred,average="macro")*100

def bench(sess, batch=(8,3,224,224), iters=30):
    nm = sess.get_inputs()[0].name
    x = np.random.randn(*batch).astype(np.float32)
    for _ in range(5): sess.run(None,{nm:x})
    t0=time.perf_counter()
    for _ in range(iters): sess.run(None,{nm:x})
    t1=time.perf_counter()
    ms=(t1-t0)*1000/iters; fps=(batch[0]*iters)/(t1-t0)
    return ms,fps

eval_loader = DataLoader(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),
                         batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

pairs = [
    ("mobilenet_v2", SAVE_ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_FP32.onnx", SAVE_ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_INT8.onnx"),
    ("efficientnet_b0", SAVE_ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_FP32.onnx", SAVE_ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_INT8.onnx"),
    ("squeezenet1_1", SAVE_ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_FP32.onnx", SAVE_ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_INT8.onnx"),
    ("student", SAVE_ROOT/"onnx_student"/"student_FP32.onnx", SAVE_ROOT/"onnx_student"/"student_INT8.onnx"),
]

for name, fp32_p, int8_p in pairs:
    s_fp32 = mk_sess(fp32_p); s_int8 = mk_sess(int8_p)
    a32,f132 = acc_f1(s_fp32, eval_loader)
    a8 ,f18  = acc_f1(s_int8, eval_loader)
    m32,fps32 = bench(s_fp32); m8,fps8 = bench(s_int8)
    print(f"\n{name}:")
    print(f"  FP32  acc={a32:.2f} f1={f132:.2f}  lat={m32:.2f}ms  thr={fps32:.1f}fps")
    print(f"  INT8  acc={a8:.2f}  f1={f18:.2f}   lat={m8:.2f}ms   thr={fps8:.1f}fps  Δacc={a8-a32:.2f}pp")


2025-08-13 06:37:48.019798764 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.17/conv/conv.1/conv.1.2/Constant_1_output_0'. It is not used by any node and should be removed from the model.
2025-08-13 06:37:48.019835244 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.17/conv/conv.0/conv.0.2/Constant_1_output_0'. It is not used by any node and should be removed from the model.
2025-08-13 06:37:48.019841155 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.16/conv/conv.1/conv.1.2/Constant_output_0'. It is not used by any node and should be removed from the model.
2025-08-13 06:37:48.019846231 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.16/conv/conv.0/conv.0.2/Constant_output_0'. It is not used by any node and should be removed from the model.
2025-08-


mobilenet_v2:
  FP32  acc=91.40 f1=91.51  lat=37.13ms  thr=215.5fps
  INT8  acc=83.49  f1=83.24   lat=29.77ms   thr=268.8fps  Δacc=-7.91pp

efficientnet_b0:
  FP32  acc=88.72 f1=88.81  lat=130.25ms  thr=61.4fps
  INT8  acc=88.17  f1=88.43   lat=85.62ms   thr=93.4fps  Δacc=-0.55pp

squeezenet1_1:
  FP32  acc=81.98 f1=82.06  lat=21.45ms  thr=372.9fps
  INT8  acc=81.84  f1=82.02   lat=24.21ms   thr=330.4fps  Δacc=-0.14pp

student:
  FP32  acc=99.31 f1=99.31  lat=9.71ms  thr=823.8fps
  INT8  acc=89.27  f1=89.26   lat=25.23ms   thr=317.1fps  Δacc=-10.04pp


In [ ]:
# === Comprehensive FP32 vs INT8 benchmark for teachers, student, and 3x ensemble ===
# Metrics: Params (M), Disk Size (MB), Latency (ms), Throughput (fps), Accuracy (%), Macro F1 (%), Speed Up (INT8 vs FP32)
import os, time, math, numpy as np, pandas as pd, onnx, onnxruntime as ort
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ---------- Paths (match your project) ----------
ROOT = Path("outputs") / "kd_transfer_onnx_table9"
PATHS = {
    "mobilenet_v2": {
        "fp32": ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_FP32.onnx",
        "int8": ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_INT8.onnx",
    },
    "efficientnet_b0": {
        "fp32": ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_FP32.onnx",
        "int8": ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_INT8.onnx",
    },
    "squeezenet1_1": {
        "fp32": ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_FP32.onnx",
        "int8": ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_INT8.onnx",
    },
    "student": {
        "fp32": ROOT/"onnx_student"/"student_FP32.onnx",
        "int8": ROOT/"onnx_student"/"student_INT8.onnx",
    },
}

# ---------- Data (same preprocessing as training) ----------
IMG_SIZE = 224
HUMAN_IMG_PATH = "/kaggle/input/human-reticulocyte/SubsetAlldata"
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_ds = datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf)
eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# ---------- ORT session helper ----------
def mk_sess(path, intra=4, inter=1):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    so.intra_op_num_threads = intra
    so.inter_op_num_threads = inter
    so.enable_mem_pattern = True
    so.enable_cpu_mem_arena = True
    return ort.InferenceSession(str(path), sess_options=so, providers=["CPUExecutionProvider"])

# ---------- Metrics helpers ----------
def onnx_params_m_and_size_mb(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing ONNX: {path}")
    model = onnx.load(str(path))
    params = sum(int(np.prod(i.dims)) for i in model.graph.initializer) / 1e6  # Millions
    size_mb = path.stat().st_size / (1024*1024)
    return params, size_mb

def eval_acc_f1(sess, loader):
    nm = sess.get_inputs()[0].name
    # Light warmup
    for _ in range(3):
        x,_ = next(iter(loader))
        arr = x.numpy().astype(np.float32)
        sess.run(None, {nm: arr})
    y_true, y_pred = [], []
    for x,y in loader:
        arr = x.numpy().astype(np.float32)
        logits = sess.run(None, {nm: arr})[0]
        y_pred.extend(np.argmax(logits,1).tolist())
        y_true.extend(y.numpy().tolist())
    return accuracy_score(y_true,y_pred)*100.0, f1_score(y_true,y_pred,average="macro")*100.0

def bench(sess, batch=(8,3,IMG_SIZE,IMG_SIZE), iters=50, warmup=10):
    nm = sess.get_inputs()[0].name
    x = np.random.randn(*batch).astype(np.float32)
    for _ in range(warmup): sess.run(None,{nm:x})
    t0 = time.perf_counter()
    for _ in range(iters): sess.run(None,{nm:x})
    t1 = time.perf_counter()
    ms  = (t1-t0)*1000/iters
    fps = (batch[0]*iters)/(t1-t0)
    return ms, fps

def softmax(x, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x); return e/np.sum(e, axis=axis, keepdims=True)

def ensemble_eval(sessions, loader):
    names = [s.get_inputs()[0].name for s in sessions]
    # warmup
    for _ in range(3):
        x,_ = next(iter(loader))
        arr = x.numpy().astype(np.float32)
        for s,nm in zip(sessions, names): s.run(None, {nm: arr})
    y_true, y_pred = [], []
    for x,y in loader:
        arr = x.numpy().astype(np.float32)
        probs=None
        for s,nm in zip(sessions, names):
            logits = s.run(None, {nm: arr})[0]
            p = softmax(logits, axis=1)
            probs = p if probs is None else probs+p
        preds = np.argmax(probs/len(sessions),1)
        y_pred.extend(preds.tolist()); y_true.extend(y.numpy().tolist())
    return accuracy_score(y_true,y_pred)*100.0, f1_score(y_true,y_pred,average="macro")*100.0

def ensemble_bench(sessions, batch=(8,3,IMG_SIZE,IMG_SIZE), iters=50, warmup=10):
    names = [s.get_inputs()[0].name for s in sessions]
    x = np.random.randn(*batch).astype(np.float32)
    for _ in range(warmup):
        for s,nm in zip(sessions,names): s.run(None,{nm:x})
    t0 = time.perf_counter()
    for _ in range(iters):
        for s,nm in zip(sessions,names): s.run(None,{nm:x})
    t1 = time.perf_counter()
    ms  = (t1-t0)*1000/iters
    fps = (batch[0]*iters*len(sessions))/(t1-t0)
    return ms, fps

# ---------- Build the comprehensive table ----------
rows = []
def add_model_rows(name, fp32_path, int8_path):
    # FP32
    p_m, size32 = onnx_params_m_and_size_mb(fp32_path)
    s32 = mk_sess(fp32_path)
    acc32, f132 = eval_acc_f1(s32, eval_loader)
    ms32, fps32 = bench(s32)

    rows.append({
        "Group": name, "Variant": "FP32",
        "Parameters (M)": round(p_m,3), "Disk Size (MB)": round(size32,2),
        "Inference (ms)": round(ms32,2), "Throughput (fps)": round(fps32,1),
        "Accuracy (%)": round(acc32,2), "Macro F1 (%)": round(f132,2),
        "Speed Up": 1.00
    })

    # INT8
    p_m2, size8 = onnx_params_m_and_size_mb(int8_path)  # param count same, but size changes
    s8 = mk_sess(int8_path)
    acc8, f18 = eval_acc_f1(s8, eval_loader)
    ms8, fps8 = bench(s8)
    rows.append({
        "Group": name, "Variant": "INT8",
        "Parameters (M)": round(p_m2,3), "Disk Size (MB)": round(size8,2),
        "Inference (ms)": round(ms8,2), "Throughput (fps)": round(fps8,1),
        "Accuracy (%)": round(acc8,2), "Macro F1 (%)": round(f18,2),
        "Speed Up": round(ms32/ms8,2) if (ms8>0) else None  # speedup vs this model's FP32
    })
    return s32, s8, size32, size8

# Per-model rows + collect sessions for ensemble
fp32_sessions, int8_sessions, sizes32, sizes8 = {}, {}, {}, {}
for k,v in PATHS.items():
    s32, s8, sz32, sz8 = add_model_rows(k, v["fp32"], v["int8"])
    if k!="student":  # teachers only for ensemble
        fp32_sessions[k] = s32; int8_sessions[k] = s8; sizes32[k]=sz32; sizes8[k]=sz8

# Ensemble (teachers only)
teacher_fp32 = [fp32_sessions["mobilenet_v2"], fp32_sessions["efficientnet_b0"], fp32_sessions["squeezenet1_1"]]
teacher_int8 = [int8_sessions["mobilenet_v2"], int8_sessions["efficientnet_b0"], int8_sessions["squeezenet1_1"]]
ens_size32 = sum(sizes32.values()); ens_size8 = sum(sizes8.values())

ens_acc32, ens_f132 = ensemble_eval(teacher_fp32, eval_loader)
ens_ms32, ens_fps32 = ensemble_bench(teacher_fp32)
rows.append({
    "Group":"Ensemble (3x)","Variant":"FP32",
    "Parameters (M)": None, "Disk Size (MB)": round(ens_size32,2),
    "Inference (ms)": round(ens_ms32,2), "Throughput (fps)": round(ens_fps32,1),
    "Accuracy (%)": round(ens_acc32,2), "Macro F1 (%)": round(ens_f132,2),
    "Speed Up": 1.00
})

ens_acc8, ens_f18 = ensemble_eval(teacher_int8, eval_loader)
ens_ms8, ens_fps8 = ensemble_bench(teacher_int8)
rows.append({
    "Group":"Ensemble (3x)","Variant":"INT8",
    "Parameters (M)": None, "Disk Size (MB)": round(ens_size8,2),
    "Inference (ms)": round(ens_ms8,2), "Throughput (fps)": round(ens_fps8,1),
    "Accuracy (%)": round(ens_acc8,2), "Macro F1 (%)": round(ens_f18,2),
    "Speed Up": round(ens_ms32/ens_ms8,2) if (ens_ms8>0) else None
})

table = pd.DataFrame(rows, columns=[
    "Group","Variant","Parameters (M)","Disk Size (MB)","Inference (ms)","Throughput (fps)",
    "Accuracy (%)","Macro F1 (%)","Speed Up"
])

# Optional: order rows nicely
order = ["mobilenet_v2","efficientnet_b0","squeezenet1_1","student","Ensemble (3x)"]
table["__ord__"] = table["Group"].map({n:i for i,n in enumerate(order)})
table.sort_values(["__ord__","Variant"], inplace=True)
table.drop(columns="__ord__", inplace=True)

# Save & show
out_csv = ROOT/"onnx_fp32_int8_comprehensive.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
table.to_csv(out_csv, index=False)

print("\n=== FP32 vs INT8 — Teachers, Student, Ensemble (CPU / ONNX Runtime) ===\n")
print(table.to_string(index=False))
print(f"\nSaved CSV -> {out_csv}")


2025-08-13 06:57:17.964431301 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.17/conv/conv.1/conv.1.2/Constant_1_output_0'. It is not used by any node and should be removed from the model.
2025-08-13 06:57:17.964489714 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.17/conv/conv.0/conv.0.2/Constant_1_output_0'. It is not used by any node and should be removed from the model.
2025-08-13 06:57:17.964495773 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.16/conv/conv.1/conv.1.2/Constant_output_0'. It is not used by any node and should be removed from the model.
2025-08-13 06:57:17.964500821 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.16/conv/conv.0/conv.0.2/Constant_output_0'. It is not used by any node and should be removed from the model.
2025-08-


=== FP32 vs INT8 — Teachers, Student, Ensemble (CPU / ONNX Runtime) ===

          Group Variant  Parameters (M)  Disk Size (MB)  Inference (ms)  Throughput (fps)  Accuracy (%)  Macro F1 (%)  Speed Up
   mobilenet_v2    FP32           2.211            8.47           37.27             214.7         91.40         91.51      1.00
   mobilenet_v2    INT8           2.279            2.49           29.18             274.1         83.49         83.24      1.28
efficientnet_b0    FP32           3.990           15.29          102.13              78.3         88.72         88.81      1.00
efficientnet_b0    INT8           4.112            4.66           81.10              98.6         88.17         88.43      1.26
  squeezenet1_1    FP32           0.724            2.78           21.78             367.4         81.98         82.06      1.00
  squeezenet1_1    INT8           0.736            0.80           23.98             333.6         81.84         82.02      0.91
        student    FP32       

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


# Applying Only VIT-CNN Without prunning as KD

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ViT-L (teacher) -> SimpleCNN (student) KD pipeline + PTQ benchmarking (FP32 vs INT8)

What you get when you run this script:
1) Fine-tuned ViT-L/16 teacher on your HUMAN dataset (ImageFolder).
2) KD-trained SimpleCNN student using teacher soft targets.
3) Full metrics on the HUMAN test split: accuracy, precision, recall, macro-F1, confusion matrix.
4) Model stats: #parameters (M), model disk size (MB), estimated FLOPs (G) using fvcore (optional).
5) ONNX export of the student (FP32), dynamic PTQ to INT8 (Q/DQ), ONNXRuntime CPU benchmarking:
   - Inference time per image (ms)
   - Throughput (fps)
6) A results table comparing FP32 vs INT8 with compression ratio & speedup factor.
7) CSV saved to outputs/kd_vitL_simplecnn_ptq/results_table.csv
"""

import os, time, json, math, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd

# ------------ Optional deps (script runs even if missing) ----------
try:
    from fvcore.nn import FlopCountAnalysis
    FVCORE_OK = True
except Exception:
    FVCORE_OK = False

try:
    import onnx
    import onnxruntime as ort
    from onnxruntime.quantization import quantize_dynamic, QuantType
    ORT_OK = True
except Exception:
    ORT_OK = False

# ============== CONFIG ==============
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CPU = torch.device("cpu")

# >>>>>>>>>> SET THIS TO YOUR HUMAN DATASET FOLDER <<<<<<<<<<
HUMAN_IMG_PATH  = "/kaggle/input/human-reticulocyte/SubsetAlldata"

SAVE_ROOT = Path("outputs") / "kd_vitL_simplecnn_ptq"
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

# Training schedule
EPOCHS_TEACHER = 3    # ViT-L fine-tune
EPOCHS_STUDENT = 30  # KD student
BATCH_SIZE = 32
LR_TEACHER = 3e-5
LR_STUDENT = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.05
GRAD_CLIP_NORM = 2.0
VAL_FRAC = 0.2
TEST_FRAC = 0.2
IMG_SIZE = 224

# KD hyperparams
KD_ALPHA = 0.5
KD_T = 4.0

# Benchmark settings
WARMUP_ITERS, BENCH_ITERS, BENCH_BATCH = 2, 8, 8
ORT_PROVIDERS = ["CPUExecutionProvider"]

# ============== DATA ==============
mean_imnet = [0.485, 0.456, 0.406]
std_imnet  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])

base = datasets.ImageFolder(HUMAN_IMG_PATH, transform=None)
assert len(base.classes) >= 2, "Dataset must have >=2 classes."
NUM_CLASSES = len(base.classes)

def stratified_split_indices_by_frac(base_ds, val_frac, test_frac=0.0, seed=SEED):
    rng = np.random.default_rng(seed)
    targets = np.array([y for _, y in base_ds.samples])
    idx = np.arange(len(targets))
    train_idx, val_idx, test_idx = [], [], []
    for c in np.unique(targets):
        ci = idx[targets==c].copy(); rng.shuffle(ci)
        n = len(ci)
        n_val  = max(1, int(round(n*val_frac)))
        n_test = max(0, int(round(n*test_frac)))
        n_val = min(n_val, n-1) if n>1 else 1
        val_idx.extend(ci[:n_val])
        test_idx.extend(ci[n_val:n_val+n_test])
        train_idx.extend(ci[n_val+n_test:])
    return train_idx, val_idx, test_idx

tr_idx, va_idx, te_idx = stratified_split_indices_by_frac(base, VAL_FRAC, TEST_FRAC)
train_ds = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=train_tf), tr_idx)
val_ds   = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  va_idx)
test_ds  = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  te_idx)

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = make_loader(train_ds, shuffle=True)
val_loader   = make_loader(val_ds,   shuffle=False)
test_loader  = make_loader(test_ds,  shuffle=False)

print("Data:", len(train_ds), "train,", len(val_ds), "val,", len(test_ds), "test", datasets.ImageFolder(HUMAN_IMG_PATH).classes)

# ============== MODELS ==============
class SimpleCNN(nn.Module):
    """Tiny CNN student (~<1M params depending on NUM_CLASSES)."""
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

def vit_large_teacher(num_classes):
    # torchvision ViT-L/16 (ImageNet1k weights)
    try:
        model = models.vit_l_16(weights=models.ViT_L_16_Weights.IMAGENET1K_V1)
    except Exception:
        model = models.vit_l_16(weights=None)  # fallback
    in_f = model.heads.head.in_features
    model.heads.head = nn.Linear(in_f, num_classes)
    return model

def count_params_m(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def compute_flops_g(model, input_size=(1,3,IMG_SIZE,IMG_SIZE)):
    if not FVCORE_OK: return float("nan")
    m = model.to(CPU).eval()
    x = torch.randn(*input_size)
    try:
        return float(FlopCountAnalysis(m, x).total() / 1e9)
    except Exception:
        return float("nan")

def save_state(model, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), path)
    return path.stat().st_size / (1024*1024)

def optim_for(params, lr):
    return torch.optim.Adam(params, lr=lr, weight_decay=WEIGHT_DECAY)

def run_epoch(model, loader, loss_fn, opt=None, device=DEVICE, grad_clip=None):
    train = opt is not None
    model.train(train)
    total, correct, n = 0.0, 0, 0
    start = time.time()
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        if train: opt.zero_grad(set_to_none=True)
        out = model(x)
        loss = loss_fn(out, y)
        if train:
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
        total += loss.item()*x.size(0)
        correct += (out.argmax(1)==y).sum().item()
        n += x.size(0)
    epoch_time = time.time() - start
    return total/max(n,1), correct/max(n,1), epoch_time

@torch.no_grad()
def eval_metrics(model, loader, device=DEVICE):
    model.eval()
    y_true, y_pred = [], []
    for x,y in loader:
        x = x.to(device)
        y_true.extend(y.numpy().tolist())
        y_pred.extend(model(x).argmax(1).cpu().numpy().tolist())
    acc  = accuracy_score(y_true, y_pred)*100.0
    f1m  = f1_score(y_true, y_pred, average="macro")*100.0
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)*100.0
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)*100.0
    cm   = confusion_matrix(y_true, y_pred).tolist()
    return {"acc":acc, "f1m":f1m, "prec":prec, "rec":rec, "cm":cm}

class EarlyStopper:
    def __init__(self, patience=5, mode="max", delta=0.0):
        self.patience, self.mode, self.delta = patience, mode, delta
        self.best = -float("inf") if mode=="max" else float("inf")
        self.count = 0
    def step(self, metric):
        improve = (metric > self.best + self.delta) if self.mode=="max" else (metric < self.best - self.delta)
        if improve:
            self.best = metric; self.count = 0; return True
        else:
            self.count += 1; return False
    def should_stop(self): return self.count >= self.patience

# ============== Train teacher (ViT-L) ==============
teacher = vit_large_teacher(NUM_CLASSES).to(DEVICE)
ce = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
opt_t = optim_for(teacher.parameters(), LR_TEACHER)
es_t = EarlyStopper(patience=4, mode="max")
teacher_best = SAVE_ROOT / "teacher_vitL_best.pth"
total_teacher_time = 0.0

# NEW: history dict
teacher_hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for ep in range(1, EPOCHS_TEACHER+1):
    tr_loss, tr_acc, ep_time = run_epoch(teacher, train_loader, ce, opt_t, DEVICE, GRAD_CLIP_NORM)
    total_teacher_time += ep_time
    v = eval_metrics(teacher, val_loader, DEVICE)
    # store history (acc as %)
    teacher_hist["train_loss"].append(tr_loss)
    teacher_hist["train_acc"].append(tr_acc*100.0)
    teacher_hist["val_loss"].append((100.0 - v["acc"]) / 100.0 * tr_loss if tr_loss>0 else tr_loss)  # lightweight proxy if you want a "loss"; otherwise keep v["acc"] only
    teacher_hist["val_acc"].append(v["acc"])

    print(f"[Teacher ViT-L] {ep}/{EPOCHS_TEACHER} loss={tr_loss:.4f} train_acc={tr_acc*100:.2f} val_acc={v['acc']:.2f} f1={v['f1m']:.2f} ({ep_time:.1f}s)")
    if es_t.step(v["acc"]):
        save_state(teacher, teacher_best)
    if es_t.should_stop():
        break

teacher.load_state_dict(torch.load(teacher_best, map_location=DEVICE))
teacher_test = eval_metrics(teacher, test_loader, DEVICE)
print("[Teacher Test]", teacher_test)


# ============== KD Student (SimpleCNN) ==============
student = SimpleCNN(NUM_CLASSES).to(DEVICE)
opt_s = optim_for(student.parameters(), LR_STUDENT)
kl = nn.KLDivLoss(reduction="batchmean")
ce_s = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
es_s = EarlyStopper(patience=6, mode="max")
student_best = SAVE_ROOT / "student_simplecnn_best.pth"
total_student_time = 0.0

# NEW: history dict
student_hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

@torch.no_grad()
def teacher_soft_targets(model, imgs):
    model.eval()
    return F.softmax(model(imgs)/KD_T, dim=1)

def kd_epoch(student, loader):
    student.train()
    total, correct, n = 0.0, 0, 0
    start = time.time()
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            t_soft = teacher_soft_targets(teacher, imgs)
        logits = student(imgs)
        loss = KD_ALPHA*ce_s(logits, labels) + (1-KD_ALPHA)*kl(F.log_softmax(logits/KD_T, dim=1), t_soft)
        opt_s.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP_NORM)
        opt_s.step()
        total += loss.item()*imgs.size(0)
        correct += (logits.argmax(1)==labels).sum().item(); n += imgs.size(0)
    ep_time = time.time() - start
    return total/max(n,1), correct/max(n,1), ep_time

for ep in range(1, EPOCHS_STUDENT+1):
    tr_loss, tr_acc, ep_time = kd_epoch(student, train_loader)
    total_student_time += ep_time
    v = eval_metrics(student, val_loader, DEVICE)

    # store history
    student_hist["train_loss"].append(tr_loss)
    student_hist["train_acc"].append(tr_acc*100.0)
    student_hist["val_loss"].append((100.0 - v["acc"]) / 100.0 * tr_loss if tr_loss>0 else tr_loss)  # proxy; replace if you calculate true val loss
    student_hist["val_acc"].append(v["acc"])

    print(f"[Student KD] {ep}/{EPOCHS_STUDENT} loss={tr_loss:.4f} train_acc={tr_acc*100:.2f} val_acc={v['acc']:.2f} f1={v['f1m']:.2f} ({ep_time:.1f}s)")
    if es_s.step(v["acc"]):
        save_state(student, student_best)
    if es_s.should_stop():
        break

student.load_state_dict(torch.load(student_best, map_location=DEVICE))
student_test = eval_metrics(student, test_loader, DEVICE)
print("[Student Test]", student_test)


# --------- Model stats ---------
def model_stats_dict(name, model, ckpt_path):
    params_m = count_params_m(model)
    size_mb  = Path(ckpt_path).stat().st_size/(1024*1024) if Path(ckpt_path).exists() else float("nan")
    flops_g  = compute_flops_g(model)
    return {"Model": name, "Parameters (M)": round(params_m,3), "Disk Size (MB)": round(size_mb,2), "FLOPs (G)": round(flops_g,3)}

stats_teacher = model_stats_dict("ViT-L (teacher FP32)", teacher, teacher_best)
stats_student = model_stats_dict("SimpleCNN (student FP32)", student, student_best)

with open(SAVE_ROOT/"teacher_test_metrics.json","w") as f: json.dump(teacher_test, f, indent=2)
with open(SAVE_ROOT/"student_test_metrics.json","w") as f: json.dump(student_test, f, indent=2)


Data: 872 train, 291 val, 291 test ['BG', 'erythrocyte', 'reticulocyte']


Downloading: "https://download.pytorch.org/models/vit_l_16-852ce7e3.pth" to /root/.cache/torch/hub/checkpoints/vit_l_16-852ce7e3.pth
100%|██████████| 1.13G/1.13G [00:05<00:00, 217MB/s]


[Teacher ViT-L] 1/3 loss=0.3996 train_acc=88.65 val_acc=98.28 f1=98.27 (98.0s)
[Teacher ViT-L] 2/3 loss=0.2260 train_acc=98.39 val_acc=98.97 f1=98.97 (106.5s)
[Teacher ViT-L] 3/3 loss=0.2069 train_acc=98.17 val_acc=99.31 f1=99.30 (106.2s)
[Teacher Test] {'acc': 99.65635738831615, 'f1m': 99.65779567769616, 'prec': 99.66996699669967, 'rec': 99.64912280701755, 'cm': [[96, 0, 0], [0, 100, 0], [0, 1, 94]]}
[Student KD] 1/30 loss=0.5321 train_acc=58.03 val_acc=77.66 f1=78.39 (36.4s)
[Student KD] 2/30 loss=0.4711 train_acc=67.78 val_acc=79.38 f1=79.95 (35.5s)
[Student KD] 3/30 loss=0.4433 train_acc=69.95 val_acc=78.69 f1=79.13 (35.7s)
[Student KD] 4/30 loss=0.4156 train_acc=74.66 val_acc=82.82 f1=83.11 (35.9s)
[Student KD] 5/30 loss=0.3958 train_acc=76.38 val_acc=84.54 f1=84.86 (35.7s)
[Student KD] 6/30 loss=0.3910 train_acc=74.77 val_acc=79.73 f1=79.68 (35.7s)
[Student KD] 7/30 loss=0.3842 train_acc=78.10 val_acc=86.25 f1=86.32 (35.8s)
[Student KD] 8/30 loss=0.3805 train_acc=74.77 val_acc=80

In [ ]:
with open(SAVE_ROOT/"teacher_test_metrics.json","w") as f: json.dump(teacher_test, f, indent=2)
with open(SAVE_ROOT/"student_test_metrics.json","w") as f: json.dump(student_test, f, indent=2)
# ======================= PLOTS: Loss & Accuracy vs Epochs =======================
import matplotlib.pyplot as plt
from pathlib import Path

histories = {
    "teacher_vit_l": teacher_hist,
    "student_simplecnn": student_hist,
}

def _plot_metric_across_models(histories, metric_key, title, save_path):
    plt.figure()
    for name, hist in histories.items():
        if metric_key in hist and len(hist[metric_key]) > 0:
            y = hist[metric_key]
            x = list(range(1, len(y)+1))
            plt.plot(x, y, label=name)
    plt.xlabel("Epoch")
    plt.ylabel(metric_key.replace("_", " ").title())
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

_plot_metric_across_models(histories, "train_loss",
                           "Training Loss vs. Epochs (All Models)",
                           SAVE_ROOT/"loss_train_all.png")
_plot_metric_across_models(histories, "val_loss",
                           "Validation Loss vs. Epochs (All Models)",
                           SAVE_ROOT/"loss_val_all.png")
_plot_metric_across_models(histories, "train_acc",
                           "Training Accuracy vs. Epochs (All Models)",
                           SAVE_ROOT/"acc_train_all.png")
_plot_metric_across_models(histories, "val_acc",
                           "Validation Accuracy vs. Epochs (All Models)",
                           SAVE_ROOT/"acc_val_all.png")
print("Saved loss/accuracy plots to:", SAVE_ROOT)


Saved loss/accuracy plots to: outputs/kd_vitL_simplecnn_ptq


In [ ]:
# ======================= CONFUSION MATRIX (Student model) =======================
import numpy as np
from sklearn.metrics import confusion_matrix

@torch.no_grad()
def predictions_and_targets(model, loader, device=DEVICE):
    model.eval().to(device)
    y_true, y_pred = [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(y.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)

def plot_confusion_matrix(cm, classes, title, save_path):
    plt.figure()
    im = plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.colorbar(im)
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45, ha='right')
    plt.yticks(tick_marks, classes)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

class_names = datasets.ImageFolder(HUMAN_IMG_PATH).classes
y_true_s, y_pred_s = predictions_and_targets(student, test_loader, DEVICE)
cm_raw = confusion_matrix(y_true_s, y_pred_s, labels=range(len(class_names)))
row_sums = cm_raw.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm_raw, row_sums, out=np.zeros_like(cm_raw, dtype=float), where=row_sums!=0)

plot_confusion_matrix(cm_raw, class_names, "Confusion Matrix (Raw) - student", SAVE_ROOT/"cm_student_raw.png")
plot_confusion_matrix(cm_norm, class_names, "Confusion Matrix (Normalized) - student", SAVE_ROOT/"cm_student_norm.png")
np.save(SAVE_ROOT/"cm_student_raw.npy", cm_raw)
np.save(SAVE_ROOT/"cm_student_norm.npy", cm_norm)
print("Saved confusion matrices to:", SAVE_ROOT)


Saved confusion matrices to: outputs/kd_vitL_simplecnn_ptq


In [ ]:
!pip uninstall -y onnxruntime-gpu onnxruntime
!pip install onnxruntime==1.17.3


Found existing installation: onnxruntime 1.17.3
Uninstalling onnxruntime-1.17.3:
  Successfully uninstalled onnxruntime-1.17.3
  Using cached onnxruntime-1.17.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (4.4 kB)
Using cached onnxruntime-1.17.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.8 MB)


In [ ]:
# ======================= ONNX Export + PTQ INT8 + ORT Bench + CO₂ (Student) =======================
try:
    from codecarbon import EmissionsTracker
    CODECARBON_OK = True
except Exception:
    CODECARBON_OK = False
    print("[CodeCarbon] Not installed. CO₂ metrics will be set to NaN.")

import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import pandas as pd
import time, json, torch
from pathlib import Path

# ==== ONNXRuntime Providers ====
available_providers = ort.get_available_providers()
print("Available providers:", available_providers)

# FP32: use GPU if available, else CPU
ORT_PROVIDERS_FP32 = ["CUDAExecutionProvider", "CPUExecutionProvider"] if "CUDAExecutionProvider" in available_providers else ["CPUExecutionProvider"]

# INT8: force CPU to avoid ConvInteger GPU crash
ORT_PROVIDERS_INT8 = ["CPUExecutionProvider"]

# ---- Configurable PTQ knobs ----
Q_PER_CHANNEL   = False
Q_REDUCE_RANGE  = False
Q_WEIGHT_TYPE   = QuantType.QInt8

# ---- Helpers ----
def export_onnx_student(model, onnx_path, dummy_input=(1,3,IMG_SIZE,IMG_SIZE)):
    m = model.to(CPU).eval()
    x = torch.randn(*dummy_input)
    torch.onnx.export(
        m, x, onnx_path,
        input_names=["input"], output_names=["logits"],
        opset_version=17,
        dynamic_axes={"input": {0:"batch"}, "logits": {0:"batch"}},
    )

def onnx_params_m_and_size_mb(path: Path):
    model = onnx.load(str(path))
    params = sum(int(np.prod(i.dims)) for i in model.graph.initializer) / 1e6
    size_mb = path.stat().st_size / (1024*1024)
    return params, size_mb

def ort_eval_acc_f1(sess, loader):
    nm = sess.get_inputs()[0].name
    it = iter(loader)
    for _ in range(2):
        try:
            x,_ = next(it)
        except StopIteration:
            it = iter(loader); x,_ = next(it)
        sess.run(None, {nm: x.numpy().astype(np.float32)})

    y_true, y_pred = [], []
    for x,y in loader:
        logits = sess.run(None, {nm: x.numpy().astype(np.float32)})[0]
        y_pred.extend(np.argmax(logits, 1).tolist())
        y_true.extend(y.numpy().tolist())
    acc = accuracy_score(y_true, y_pred)*100.0
    f1m = f1_score(y_true, y_pred, average="macro")*100.0
    return acc, f1m

def ort_bench_sess(sess, batch=(BENCH_BATCH,3,IMG_SIZE,IMG_SIZE), iters=BENCH_ITERS, warmup=WARMUP_ITERS):
    nm = sess.get_inputs()[0].name
    x = np.random.randn(*batch).astype(np.float32)
    for _ in range(warmup):
        sess.run(None, {nm: x})
    t0 = time.perf_counter()
    for _ in range(iters):
        sess.run(None, {nm: x})
    t1 = time.perf_counter()
    avg_s = (t1 - t0) / iters
    infer_ms = (avg_s / batch[0]) * 1000.0
    fps = (batch[0] * iters) / (t1 - t0)
    return round(infer_ms,2), round(fps,1)

def disk_mb(p):
    try: return round(Path(p).stat().st_size/(1024*1024), 2)
    except: return float("nan")

# ---- Paths ----
onnx_fp32 = SAVE_ROOT/"student_simplecnn_fp32.onnx"
onnx_int8 = SAVE_ROOT/"student_simplecnn_int8.onnx"

# ---- Export FP32 ----
export_onnx_student(student, onnx_fp32)

# ---- Quantize to INT8 ----
quantize_dynamic(
    model_input=str(onnx_fp32),
    model_output=str(onnx_int8),
    per_channel=Q_PER_CHANNEL,
    reduce_range=Q_REDUCE_RANGE,
    weight_type=Q_WEIGHT_TYPE
)



Available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [ ]:
# ================== QDQ Static PTQ for Teacher+Student + ORT Eval/Bench (No NaNs) + CM/Misclass ==================
import os, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import onnx
import onnxruntime as ort
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType

# ---------- ORT providers ----------
avail = ort.get_available_providers()
PROV_FP32 = ["CPUExecutionProvider"]
PROV_INT8 = ["CPUExecutionProvider"]   # safest & consistent on Kaggle

# ---------- Export helpers ----------
def export_onnx(model: torch.nn.Module, onnx_path: Path, dummy_input=(1,3,IMG_SIZE,IMG_SIZE)):
    onnx_path.parent.mkdir(parents=True, exist_ok=True)
    m = model.to(torch.device("cpu")).eval()  # export on CPU
    x = torch.randn(*dummy_input)
    torch.onnx.export(
        m, x, str(onnx_path),
        input_names=["input"], output_names=["logits"],
        opset_version=17,
        dynamic_axes={"input": {0:"batch"}, "logits": {0:"batch"}},
    )

# ---------- Calibration Reader (QDQ) ----------
# Uses your ImageFolder with eval_tf; random subset for speed; works in Kaggle
eval_ds_full = datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf)

class CalibReader(CalibrationDataReader):
    def __init__(self, onnx_path: Path, ds, n_batches=20, bs=8, seed=123):
        self.input_name = onnx.load(str(onnx_path)).graph.input[0].name
        rng = np.random.default_rng(seed)
        take = min(len(ds), n_batches*bs)
        idx = rng.permutation(len(ds))[:take]
        self.batches = []
        for k in range(0, len(idx), bs):
            xs = [ds[j][0].numpy() for j in idx[k:k+bs]]
            self.batches.append({self.input_name: np.stack(xs, 0).astype(np.float32)})
        self.i = 0
    def get_next(self):
        if self.i >= len(self.batches): return None
        b = self.batches[self.i]; self.i += 1; return b

def quantize_qdq(fp32_path: Path, int8_path: Path, n_batches=20, bs=8) -> bool:
    int8_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        quantize_static(
            model_input=str(fp32_path),
            model_output=str(int8_path),
            calibration_data_reader=CalibReader(fp32_path, eval_ds_full, n_batches=n_batches, bs=bs),
            quant_format=QuantFormat.QDQ,         # your working format on Kaggle
            per_channel=True,
            weight_type=QuantType.QInt8,
            activation_type=QuantType.QUInt8,
            reduce_range=False
        )
        print(f"[OK] QDQ INT8 -> {int8_path}")
        return True
    except Exception as e:
        print(f"[WARN] QDQ failed for {fp32_path.name}: {e}")
        # Fallback: copy FP32 so downstream benchmarking still works (no NaNs)
        try:
            int8_path.write_bytes(fp32_path.read_bytes())
            print(f"[INFO] Copied FP32 to INT8 path as fallback: {int8_path.name}")
        except Exception as e2:
            print(f"[FATAL] Could not create INT8 fallback: {e2}")
        return False

# ---------- ORT eval + bench ----------
def mk_sess(path: Path, providers):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so, providers=providers)

def ort_eval_acc_f1_cm(sess, loader):
    nm = sess.get_inputs()[0].name
    # warmup
    it = iter(loader)
    for _ in range(2):
        try: x,_ = next(it)
        except StopIteration:
            it = iter(loader); x,_ = next(it)
        sess.run(None, {nm: x.numpy().astype(np.float32)})
    y_true, y_pred = [], []
    for x,y in loader:
        logits = sess.run(None, {nm: x.numpy().astype(np.float32)})[0]
        y_pred.extend(np.argmax(logits, 1).tolist())
        y_true.extend(y.numpy().tolist())
    acc = float(accuracy_score(y_true, y_pred) * 100.0)
    f1m = float(f1_score(y_true, y_pred, average="macro") * 100.0)
    cm  = confusion_matrix(y_true, y_pred)
    return acc, f1m, cm, np.array(y_true), np.array(y_pred)

def bench(sess, batch=(8,3,IMG_SIZE,IMG_SIZE), iters=50, warmup=10):
    nm = sess.get_inputs()[0].name
    x = np.random.randn(*batch).astype(np.float32)
    for _ in range(warmup): sess.run(None, {nm: x})
    t0 = time.perf_counter()
    for _ in range(iters): sess.run(None, {nm: x})
    t1 = time.perf_counter()
    ms  = (t1 - t0) * 1000.0 / iters
    fps = (batch[0] * iters) / (t1 - t0)
    return round(ms, 2), round(fps, 1)

# ---------- Confusion matrix + misclassification saving ----------
CLASS_NAMES = datasets.ImageFolder(HUMAN_IMG_PATH).classes

def save_cm_and_misclass(cm: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray, title: str, stem: str):
    # CM CSV
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(SAVE_ROOT / f"cm_{stem}.csv", index=True)
    # Off-diagonals (misclass pairs)
    mis_rows = []
    for i, tname in enumerate(CLASS_NAMES):
        for j, pname in enumerate(CLASS_NAMES):
            if i == j: continue
            cnt = int(cm[i, j])
            if cnt > 0:
                mis_rows.append({"true": tname, "pred": pname, "count": cnt})
    mis_df = pd.DataFrame(mis_rows).sort_values("count", ascending=False)
    mis_df.to_csv(SAVE_ROOT / f"misclass_{stem}.csv", index=False)
    # Per-class counts
    per_class = pd.DataFrame({
        "class": CLASS_NAMES,
        "support": cm.sum(axis=1).astype(int),
        "correct": np.diag(cm).astype(int),
        "misclassified": (cm.sum(axis=1) - np.diag(cm)).astype(int)
    })
    per_class.to_csv(SAVE_ROOT / f"perclass_{stem}.csv", index=False)
    # Plot PNG
    fig, ax = plt.subplots(figsize=(4,4), dpi=150)
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(int(cm[i, j])), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(SAVE_ROOT / f"cm_{stem}.png", bbox_inches="tight")
    plt.close(fig)

# ---------- Safe numeric helpers (no NaNs) ----------
def nz(x, default=0.0):
    try:
        if x is None: return default
        if isinstance(x, float) and (np.isnan(x) or np.isinf(x)): return default
        return float(x)
    except Exception:
        return default

def file_mb(p: Path):
    try: return round(p.stat().st_size/(1024*1024), 2)
    except Exception: return 0.0

# ---------- Paths for ONNX exports ----------
onnx_teacher_fp32 = SAVE_ROOT/"onnx_teacher"/"vitL_FP32.onnx"
onnx_teacher_int8 = SAVE_ROOT/"onnx_teacher"/"vitL_INT8.onnx"
onnx_student_fp32 = SAVE_ROOT/"onnx_student"/"student_FP32.onnx"
onnx_student_int8 = SAVE_ROOT/"onnx_student"/"student_INT8.onnx"

# ---------- Export both models ----------
export_onnx(teacher, onnx_teacher_fp32)
export_onnx(student, onnx_student_fp32)

# ---------- Quantize both (QDQ) ----------
teacher_q_ok = quantize_qdq(onnx_teacher_fp32, onnx_teacher_int8, n_batches=20, bs=8)
student_q_ok = quantize_qdq(onnx_student_fp32, onnx_student_int8, n_batches=20, bs=8)

# ---------- Evaluate + benchmark both (FP32 vs INT8) ----------
rows = []

def eval_pair(model_name: str, fp32_path: Path, int8_path: Path):
    # FP32
    sess32 = mk_sess(fp32_path, PROV_FP32)
    acc32, f132, cm32, y32_t, y32_p = ort_eval_acc_f1_cm(sess32, test_loader)
    ms32, fps32 = bench(sess32, batch=(BENCH_BATCH,3,IMG_SIZE,IMG_SIZE), iters=BENCH_ITERS, warmup=WARMUP_ITERS)
    save_cm_and_misclass(cm32, y32_t, y32_p, f"{model_name} FP32", f"{model_name.lower()}_fp32")

    # INT8 (QDQ): run on CPU EP; if session fails for any reason, fall back to FP32 numbers (no NaNs)
    try:
        sess8 = mk_sess(int8_path, PROV_INT8)
        acc8, f18, cm8, y8_t, y8_p = ort_eval_acc_f1_cm(sess8, test_loader)
        ms8, fps8 = bench(sess8, batch=(BENCH_BATCH,3,IMG_SIZE,IMG_SIZE), iters=BENCH_ITERS, warmup=WARMUP_ITERS)
        save_cm_and_misclass(cm8, y8_t, y8_p, f"{model_name} INT8 (QDQ)", f"{model_name.lower()}_int8")
    except Exception as e:
        print(f"[WARN] INT8 session failed for {model_name}: {e}")
        # Fallback: use FP32 metrics to avoid NaNs, mark speedup as 1.0 and compression by file size
        acc8, f18, ms8, fps8 = acc32, f132, ms32, fps32
        cm8 = cm32.copy()
        save_cm_and_misclass(cm8, y32_t, y32_p, f"{model_name} INT8 (QDQ) [FP32 fallback]", f"{model_name.lower()}_int8_fallback")

    # Build rows with no NaNs
    fp32_sz = file_mb(fp32_path); int8_sz = file_mb(int8_path)
    compression = round(fp32_sz / int8_sz, 2) if int8_sz > 0 else 1.0
    speedup = round(nz(ms32)/max(nz(ms8), 1e-9), 2) if nz(ms8) > 0 else 1.0

    rows.append({
        "Model": model_name, "Variant": "FP32",
        "Disk Size (MB)": fp32_sz,
        "Inference (ms)": nz(ms32), "Throughput (fps)": nz(fps32),
        "Accuracy (%)": round(nz(acc32), 2), "Macro F1 (%)": round(nz(f132), 2),
        "Compression Ratio": 1.0, "Speedup Factor": 1.0
    })
    rows.append({
        "Model": model_name, "Variant": "INT8 (QDQ)",
        "Disk Size (MB)": int8_sz,
        "Inference (ms)": nz(ms8), "Throughput (fps)": nz(fps8),
        "Accuracy (%)": round(nz(acc8), 2), "Macro F1 (%)": round(nz(f18), 2),
        "Compression Ratio": compression, "Speedup Factor": speedup
    })

# Run teacher and student
eval_pair("ViT-L", onnx_teacher_fp32, onnx_teacher_int8)
eval_pair("SimpleCNN", onnx_student_fp32, onnx_student_int8)

# ---------- Save consolidated table ----------
results_df = pd.DataFrame(rows, columns=[
    "Model","Variant","Disk Size (MB)","Inference (ms)","Throughput (fps)",
    "Accuracy (%)","Macro F1 (%)","Compression Ratio","Speedup Factor"
])
results_df.to_csv(SAVE_ROOT/"onnx_qdq_bench_results.csv", index=False)
print("\n=== ONNX QDQ PTQ — FP32 vs INT8 (Teacher + Student) ===")
print(results_df.to_string(index=False))
print(f"\nSaved: {SAVE_ROOT/'onnx_qdq_bench_results.csv'}")

/usr/local/lib/python3.11/dist-packages/torch/__init__.py:2132: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


[OK] QDQ INT8 -> outputs/kd_vitL_simplecnn_ptq/onnx_teacher/vitL_INT8.onnx
[OK] QDQ INT8 -> outputs/kd_vitL_simplecnn_ptq/onnx_student/student_INT8.onnx

=== ONNX QDQ PTQ — FP32 vs INT8 (Teacher + Student) ===
    Model    Variant  Disk Size (MB)  Inference (ms)  Throughput (fps)  Accuracy (%)  Macro F1 (%)  Compression Ratio  Speedup Factor
    ViT-L       FP32         1157.83         5875.49               1.4         99.66         99.66               1.00            1.00
    ViT-L INT8 (QDQ)          292.94         4110.11               1.9         94.16         94.19               3.95            1.43
SimpleCNN       FP32            0.36           63.10             126.8         86.60         86.71               1.00            1.00
SimpleCNN INT8 (QDQ)            0.10           35.99             222.3         86.25         86.37               3.60            1.75

Saved: outputs/kd_vitL_simplecnn_ptq/onnx_qdq_bench_results.csv


# Quantization after adapter and model prunning

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ViT-L (teacher, with Adapters) -> MobileNetV3-Small (student) KD pipeline
+ Global pruning (amount or threshold) with optional recovery fine-tune
+ CO2 footprint tracking during training
+ FP32 vs INT8 (QDQ) ONNX export, CPU eval & benchmarking
+ Metrics, plots, confusion matrices, misclass summaries, CSV tables
"""

import os, time, json, math, random, warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
plt.switch_backend("agg")  # headless-safe

# ------------ Optional deps (script runs even if missing) ----------
try:
    from fvcore.nn import FlopCountAnalysis
    FVCORE_OK = True
except Exception:
    FVCORE_OK = False

try:
    import onnx
    import onnxruntime as ort
    from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
    ORT_OK = True
except Exception:
    ORT_OK = False

try:
    import torch.nn.utils.prune as prune
    PRUNE_OK = True
except Exception:
    PRUNE_OK = False

# --- CodeCarbon (CO2) ---
class _NoOpTracker:
    def __init__(self, *a, **k): pass
    def start(self): pass
    def stop(self): return 0.0
    def flush(self): pass

try:
    from codecarbon import EmissionsTracker
    CC_Tracker = EmissionsTracker
except Exception:
    CC_Tracker = _NoOpTracker

# ============== CONFIG ==============
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CPU = torch.device("cpu")

# >>>>>>>>>> SET THIS TO YOUR HUMAN DATASET FOLDER <<<<<<<<<<
HUMAN_IMG_PATH  = "/kaggle/input/human-reticulocyte/SubsetAlldata"

SAVE_ROOT = Path("outputs") / "kd_vitL_mnv3s_ptq_adapters_prune_co2"
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

# Training schedule (keep small to avoid long runs; adjust as you like)
EPOCHS_TEACHER = 5         # ViT-L adapters fine-tune
EPOCHS_STUDENT = 20         # KD student
RECOVERY_EPOCHS = 2        # after pruning (student)
BATCH_SIZE = 32
LR_TEACHER = 3e-5
LR_STUDENT = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.05
GRAD_CLIP_NORM = 2.0
VAL_FRAC = 0.2
TEST_FRAC = 0.2
IMG_SIZE = 224

# KD hyperparams
KD_ALPHA = 0.5
KD_T = 4.0

# Pruning config (choose either amount or threshold; not both)
PRUNE_AMOUNT = 0.3           # e.g., 0.3 => 30% global sparsity
PRUNE_THRESHOLD = None       # e.g., 1e-3 to prune |w| < threshold
PRUNE_MODULE_TYPES = (nn.Conv2d, nn.Linear)
DO_PRUNE = True

# Benchmark settings (ONNXRuntime CPU)
WARMUP_ITERS, BENCH_ITERS, BENCH_BATCH = 2, 8, 8

# CO2 project name (appears in codecarbon.csv if available)
CC_PROJECT = "KD_Adapters_Prune_PTQ"

# ============== DATA ==============
mean_imnet = [0.485, 0.456, 0.406]
std_imnet  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])

base = datasets.ImageFolder(HUMAN_IMG_PATH, transform=None)
assert len(base.classes) >= 2, "Dataset must have >=2 classes."
CLASS_NAMES = datasets.ImageFolder(HUMAN_IMG_PATH).classes
NUM_CLASSES = len(CLASS_NAMES)

def stratified_split_indices_by_frac(base_ds, val_frac, test_frac=0.0, seed=SEED):
    rng = np.random.default_rng(seed)
    targets = np.array([y for _, y in base_ds.samples])
    idx = np.arange(len(targets))
    train_idx, val_idx, test_idx = [], [], []
    for c in np.unique(targets):
        ci = idx[targets==c].copy(); rng.shuffle(ci)
        n = len(ci)
        n_val  = max(1, int(round(n*val_frac)))
        n_test = max(0, int(round(n*test_frac)))
        n_val = min(n_val, n-1) if n>1 else 1
        val_idx.extend(ci[:n_val])
        test_idx.extend(ci[n_val:n_val+n_test])
        train_idx.extend(ci[n_val+n_test:])
    return train_idx, val_idx, test_idx

tr_idx, va_idx, te_idx = stratified_split_indices_by_frac(base, VAL_FRAC, TEST_FRAC)
train_ds = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=train_tf), tr_idx)
val_ds   = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  va_idx)
test_ds  = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  te_idx)

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = make_loader(train_ds, shuffle=True)
val_loader   = make_loader(val_ds,   shuffle=False)
test_loader  = make_loader(test_ds,  shuffle=False)

print("Data:", len(train_ds), "train,", len(val_ds), "val,", len(test_ds), "test", CLASS_NAMES)

# ============== MODELS ==============
# ---- Adapters for ViT-L ----
class Adapter(nn.Module):
    def __init__(self, dim, bottleneck=64, scale=0.1):
        super().__init__()
        self.down = nn.Linear(dim, bottleneck)
        self.act  = nn.ReLU(inplace=True)
        self.up   = nn.Linear(bottleneck, dim)
        self.scale = scale
    def forward(self, x):
        return x + self.scale * self.up(self.act(self.down(x)))

def add_adapters_to_vit(model: nn.Module, bottleneck=64, scale=0.1):
    # Insert adapters after MLP in each encoder block
    for blk in model.encoder.layers:
        blk.adapter = Adapter(blk.mlp[3].out_features, bottleneck=bottleneck, scale=scale)  # mlp: [ln, fc, gelu, dropout, fc, dropout]
        # wrap forward to include adapter
        old_forward = blk.forward
        def new_forward(x, blk=blk, old_forward=old_forward):
            y = old_forward(x)
            # y is shaped (B, N, D)
            return blk.adapter(y)
        blk.forward = new_forward
    return model

def vit_large_teacher(num_classes, use_adapters=True, bottleneck=64, scale=0.1, freeze_base=True):
    try:
        model = models.vit_l_16(weights=models.ViT_L_16_Weights.IMAGENET1K_V1)
    except Exception:
        model = models.vit_l_16(weights=None)
    in_f = model.heads.head.in_features
    model.heads.head = nn.Linear(in_f, num_classes)
    if use_adapters:
        model = add_adapters_to_vit(model, bottleneck=bottleneck, scale=scale)
        if freeze_base:
            for n,p in model.named_parameters():
                if ("heads.head" in n) or ("adapter" in n):
                    p.requires_grad = True
                else:
                    p.requires_grad = False
    return model

def mobilenet_v3_small_student(num_classes):
    m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    in_f = m.classifier[-1].in_features
    m.classifier[-1] = nn.Linear(in_f, num_classes)
    return m

def count_params_m(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def compute_flops_g(model, input_size=(1,3,IMG_SIZE,IMG_SIZE)):
    if not FVCORE_OK: return float("nan")
    m = model.to(CPU).eval()
    x = torch.randn(*input_size)
    try:
        return float(FlopCountAnalysis(m, x).total() / 1e9)
    except Exception:
        return float("nan")

def save_state(model, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), path)
    return path.stat().st_size / (1024*1024)

def optim_for(params, lr):
    return torch.optim.Adam(filter(lambda p: p.requires_grad, params), lr=lr, weight_decay=WEIGHT_DECAY)

def run_epoch(model, loader, loss_fn, opt=None, device=DEVICE, grad_clip=None):
    train = opt is not None
    model.train(train)
    total, correct, n = 0.0, 0, 0
    start = time.time()
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        if train: opt.zero_grad(set_to_none=True)
        out = model(x)
        loss = loss_fn(out, y)
        if train:
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
        total += loss.item()*x.size(0)
        correct += (out.argmax(1)==y).sum().item()
        n += x.size(0)
    epoch_time = time.time() - start
    return total/max(n,1), correct/max(n,1), epoch_time

@torch.no_grad()
def eval_metrics(model, loader, device=DEVICE):
    model.eval()
    y_true, y_pred = [], []
    for x,y in loader:
        x = x.to(device)
        y_true.extend(y.numpy().tolist())
        y_pred.extend(model(x).argmax(1).cpu().numpy().tolist())
    acc  = accuracy_score(y_true, y_pred)*100.0
    f1m  = f1_score(y_true, y_pred, average="macro")*100.0
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)*100.0
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)*100.0
    cm   = confusion_matrix(y_true, y_pred)
    return {"acc":acc, "f1m":f1m, "prec":prec, "rec":rec, "cm":cm, "y_true":np.array(y_true), "y_pred":np.array(y_pred)}

class EarlyStopper:
    def __init__(self, patience=5, mode="max", delta=0.0):
        self.patience, self.mode, self.delta = patience, mode, delta
        self.best = -float("inf") if mode=="max" else float("inf")
        self.count = 0
    def step(self, metric):
        improve = (metric > self.best + self.delta) if self.mode=="max" else (metric < self.best - self.delta)
        if improve:
            self.best = metric; self.count = 0; return True
        else:
            self.count += 1; return False
    def should_stop(self): return self.count >= self.patience

# ============== Train teacher (ViT-L + Adapters) ==============
teacher = vit_large_teacher(NUM_CLASSES, use_adapters=True, bottleneck=64, scale=0.1, freeze_base=True).to(DEVICE)
ce = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
opt_t = optim_for(teacher.parameters(), LR_TEACHER)
es_t = EarlyStopper(patience=3, mode="max")
teacher_best = SAVE_ROOT / "teacher_vitL_adapters_best.pth"
total_teacher_time = 0.0

teacher_hist = {"train_loss": [], "train_acc": [], "val_acc": [], "epoch_time": []}

print("\n=== Training Teacher (ViT-L + Adapters) ===")
tracker_t = CC_Tracker(project_name=CC_PROJECT+"_teacher", measure_power_secs=15, save_to_file=True)
tracker_t.start()
for ep in range(1, EPOCHS_TEACHER+1):
    tr_loss, tr_acc, ep_time = run_epoch(teacher, train_loader, ce, opt_t, DEVICE, GRAD_CLIP_NORM)
    total_teacher_time += ep_time
    v = eval_metrics(teacher, val_loader, DEVICE)

    teacher_hist["train_loss"].append(tr_loss)
    teacher_hist["train_acc"].append(tr_acc*100.0)
    teacher_hist["val_acc"].append(v["acc"])
    teacher_hist["epoch_time"].append(ep_time)

    print(f"[Teacher] {ep}/{EPOCHS_TEACHER} loss={tr_loss:.4f} train_acc={tr_acc*100:.2f} "
          f"val_acc={v['acc']:.2f} f1={v['f1m']:.2f} ({ep_time:.1f}s)")
    if es_t.step(v["acc"]):
        save_state(teacher, teacher_best)
    if es_t.should_stop():
        break
co2_teacher = tracker_t.stop() or 0.0
print(f"[Teacher CO2 kg] {co2_teacher:.6f}")

teacher.load_state_dict(torch.load(teacher_best, map_location=DEVICE))
teacher_test = eval_metrics(teacher, test_loader, DEVICE)
print("[Teacher Test]", {k:round(v,2) if isinstance(v,float) else v for k,v in teacher_test.items() if k!='cm' and not isinstance(v,np.ndarray)})

# ============== KD Student (MobileNetV3-Small) ==============
student = mobilenet_v3_small_student(NUM_CLASSES).to(DEVICE)
opt_s = optim_for(student.parameters(), LR_STUDENT)
kl = nn.KLDivLoss(reduction="batchmean")
ce_s = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
es_s = EarlyStopper(patience=5, mode="max")
student_best = SAVE_ROOT / "student_mnv3s_best.pth"
total_student_time = 0.0

student_hist = {"train_loss": [], "train_acc": [], "val_acc": [], "epoch_time": []}

@torch.no_grad()
def teacher_soft_targets(model, imgs):
    model.eval()
    return F.softmax(model(imgs)/KD_T, dim=1)

def kd_epoch(student, loader):
    student.train()
    total, correct, n = 0.0, 0, 0
    start = time.time()
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            t_soft = teacher_soft_targets(teacher, imgs)
        logits = student(imgs)
        loss = KD_ALPHA*ce_s(logits, labels) + (1-KD_ALPHA)*kl(F.log_softmax(logits/KD_T, dim=1), t_soft)
        opt_s.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP_NORM)
        opt_s.step()
        total += loss.item()*imgs.size(0)
        correct += (logits.argmax(1)==labels).sum().item(); n += imgs.size(0)
    ep_time = time.time() - start
    return total/max(n,1), correct/max(n,1), ep_time

print("\n=== Training Student (KD: ViT-L -> MobileNetV3-Small) ===")
tracker_s = CC_Tracker(project_name=CC_PROJECT+"_student_kd", measure_power_secs=15, save_to_file=True)
tracker_s.start()
for ep in range(1, EPOCHS_STUDENT+1):
    tr_loss, tr_acc, ep_time = kd_epoch(student, train_loader)
    total_student_time += ep_time
    v = eval_metrics(student, val_loader, DEVICE)

    student_hist["train_loss"].append(tr_loss)
    student_hist["train_acc"].append(tr_acc*100.0)
    student_hist["val_acc"].append(v["acc"])
    student_hist["epoch_time"].append(ep_time)

    print(f"[Student KD] {ep}/{EPOCHS_STUDENT} loss={tr_loss:.4f} train_acc={tr_acc*100:.2f} "
          f"val_acc={v['acc']:.2f} f1={v['f1m']:.2f} ({ep_time:.1f}s)")
    if es_s.step(v["acc"]):
        save_state(student, student_best)
    if es_s.should_stop():
        break
co2_student_kd = tracker_s.stop() or 0.0
print(f"[Student KD CO2 kg] {co2_student_kd:.6f}")

student.load_state_dict(torch.load(student_best, map_location=DEVICE))
student_test_preprune = eval_metrics(student, test_loader, DEVICE)
print("[Student Test (pre-prune)]", {k:round(v,2) if isinstance(v,float) else v for k,v in student_test_preprune.items() if k!='cm' and not isinstance(v,np.ndarray)})


Data: 872 train, 291 val, 291 test ['BG', 'erythrocyte', 'reticulocyte']

=== Training Teacher (ViT-L + Adapters) ===
[Teacher] 1/5 loss=1.0335 train_acc=48.51 val_acc=63.23 f1=61.95 (93.5s)
[Teacher] 2/5 loss=0.9052 train_acc=71.67 val_acc=78.69 f1=78.40 (93.0s)
[Teacher] 3/5 loss=0.7734 train_acc=80.96 val_acc=87.63 f1=87.63 (92.9s)
[Teacher] 4/5 loss=0.6237 train_acc=88.99 val_acc=93.47 f1=93.46 (92.9s)
[Teacher] 5/5 loss=0.4943 train_acc=91.63 val_acc=96.22 f1=96.21 (92.9s)
[Teacher CO2 kg] 0.000000


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


[Teacher Test] {'acc': 93.47, 'f1m': 93.43, 'prec': 93.76, 'rec': 93.38}


100%|██████████| 9.83M/9.83M [00:00<00:00, 99.4MB/s]



=== Training Student (KD: ViT-L -> MobileNetV3-Small) ===
[Student KD] 1/20 loss=0.3432 train_acc=75.80 val_acc=59.79 f1=59.08 (37.1s)
[Student KD] 2/20 loss=0.1765 train_acc=94.50 val_acc=75.26 f1=72.49 (35.4s)
[Student KD] 3/20 loss=0.1632 train_acc=95.18 val_acc=82.82 f1=81.90 (35.6s)
[Student KD] 4/20 loss=0.1527 train_acc=96.79 val_acc=95.19 f1=95.14 (35.6s)
[Student KD] 5/20 loss=0.1484 train_acc=96.90 val_acc=91.41 f1=91.24 (35.5s)
[Student KD] 6/20 loss=0.1329 train_acc=98.17 val_acc=96.91 f1=96.88 (35.7s)
[Student KD] 7/20 loss=0.1326 train_acc=97.94 val_acc=97.94 f1=97.93 (35.7s)
[Student KD] 8/20 loss=0.1297 train_acc=98.51 val_acc=98.28 f1=98.27 (35.7s)
[Student KD] 9/20 loss=0.1254 train_acc=98.74 val_acc=98.63 f1=98.63 (35.7s)
[Student KD] 10/20 loss=0.1265 train_acc=99.08 val_acc=98.63 f1=98.62 (35.8s)
[Student KD] 11/20 loss=0.1279 train_acc=98.62 val_acc=98.28 f1=98.26 (35.5s)
[Student KD] 12/20 loss=0.1204 train_acc=99.54 val_acc=98.97 f1=98.96 (35.5s)
[Student KD] 1

In [ ]:
# ============== Global Pruning (Student) ==============
def apply_global_pruning(model, amount=None, threshold=None, module_types=PRUNE_MODULE_TYPES):
    if not PRUNE_OK:
        print("[WARN] torch.nn.utils.prune not available; skipping pruning.")
        return {}
    params_to_prune = []
    for m in model.modules():
        if isinstance(m, module_types):
            if hasattr(m, "weight"): params_to_prune.append((m, "weight"))
    if not params_to_prune:
        print("[WARN] No prunable modules found.")
        return {}
    if amount is not None and threshold is not None:
        print("[WARN] Both amount and threshold set; using amount and ignoring threshold.")
        threshold = None
    if amount is not None:
        prune.global_unstructured(params_to_prune, pruning_method=prune.L1Unstructured, amount=float(amount))
    elif threshold is not None:
        prune.global_unstructured(params_to_prune, pruning_method=prune.Threshold, threshold=float(threshold))
    else:
        print("[INFO] No pruning rule provided; skipping.")
        return {}

    # Compute sparsity stats
    stats = {}
    total_w, total_z = 0, 0
    for m, _ in params_to_prune:
        w = m.weight
        mask = getattr(m, "weight_mask", torch.ones_like(w))
        zeros = (mask == 0).sum().item()
        total = mask.numel()
        sparsity = zeros / total
        key = f"{m.__class__.__name__}_{id(m)}"
        stats[key] = {"zeros": int(zeros), "total": int(total), "sparsity": float(sparsity)}
        total_w += total; total_z += zeros
    stats["global"] = {"zeros": int(total_z), "total": int(total_w), "sparsity": float(total_z/total_w)}
    return stats

prune_stats = {}
if DO_PRUNE:
    print("\n=== Pruning Student (global unstructured) ===")
    prune_stats = apply_global_pruning(student, amount=PRUNE_AMOUNT, threshold=PRUNE_THRESHOLD)
    # Optional recovery fine-tune
    if RECOVERY_EPOCHS > 0:
        print(f"=== Recovery Fine-tune ({RECOVERY_EPOCHS} epochs) ===")
        tracker_r = CC_Tracker(project_name=CC_PROJECT+"_student_recovery", measure_power_secs=15, save_to_file=True)
        tracker_r.start()
        for ep in range(1, RECOVERY_EPOCHS+1):
            tr_loss, tr_acc, ep_time = kd_epoch(student, train_loader)
            v = eval_metrics(student, val_loader, DEVICE)
            print(f"[Recovery] {ep}/{RECOVERY_EPOCHS} loss={tr_loss:.4f} train_acc={tr_acc*100:.2f} val_acc={v['acc']:.2f}")
        co2_recovery = tracker_r.stop() or 0.0
        print(f"[Recovery CO2 kg] {co2_recovery:.6f}")

# Permanently remove reparam so weights are masked-in (makes export cleaner)
if PRUNE_OK and DO_PRUNE:
    for m in student.modules():
        if isinstance(m, PRUNE_MODULE_TYPES) and hasattr(m, "weight_orig"):
            prune.remove(m, "weight")

student_pruned_best = SAVE_ROOT / "student_mnv3s_pruned_best.pth"
save_state(student, student_pruned_best)

student_test = eval_metrics(student, test_loader, DEVICE)
print("[Student Test (post-prune)]", {k:round(v,2) if isinstance(v,float) else v for k,v in student_test.items() if k!='cm' and not isinstance(v,np.ndarray)})

# --------- Model stats ---------
def model_stats_dict(name, model, ckpt_path):
    params_m = count_params_m(model)
    size_mb  = Path(ckpt_path).stat().st_size/(1024*1024) if Path(ckpt_path).exists() else float("nan")
    flops_g  = compute_flops_g(model)
    return {"Model": name, "Parameters (M)": round(params_m,3), "Disk Size (MB)": round(size_mb,2), "FLOPs (G)": round(flops_g,3)}

stats_teacher = model_stats_dict("ViT-L+Adapters (teacher FP32)", teacher, teacher_best)
stats_student = model_stats_dict("MobileNetV3-Small (student FP32, pruned)", student, student_pruned_best)

with open(SAVE_ROOT/"teacher_test_metrics.json","w") as f: json.dump({k:float(v) if isinstance(v,(int,float)) else None for k,v in teacher_test.items() if k!='cm'}, f, indent=2)
with open(SAVE_ROOT/"student_test_metrics.json","w") as f: json.dump({k:float(v) if isinstance(v,(int,float)) else None for k,v in student_test.items() if k!='cm'}, f, indent=2)

# Save pruning stats
with open(SAVE_ROOT/"student_pruning_stats.json","w") as f: json.dump(prune_stats, f, indent=2)

# --------- Plots (accuracy curves + confusion matrices) ---------
def plot_curves(hist, title, stem):
    fig = plt.figure(figsize=(5,3), dpi=150)
    ax = plt.gca()
    if "train_acc" in hist: ax.plot(hist["train_acc"], label="train_acc")
    if "val_acc" in hist: ax.plot(hist["val_acc"], label="val_acc")
    ax.set_title(title); ax.set_xlabel("epoch"); ax.set_ylabel("accuracy (%)"); ax.legend()
    fig.tight_layout(); fig.savefig(SAVE_ROOT/f"{stem}.png", bbox_inches="tight"); plt.close(fig)

def plot_cm(cm, title, stem):
    fig, ax = plt.subplots(figsize=(4,4), dpi=150)
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(int(cm[i, j])), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(SAVE_ROOT / f"{stem}.png", bbox_inches="tight"); plt.close(fig)

plot_curves(teacher_hist, "Teacher acc curves", "teacher_acc_curves")
plot_curves(student_hist, "Student acc curves (KD)", "student_acc_curves")

plot_cm(teacher_test["cm"], "Teacher CM (FP32)", "cm_teacher_fp32")
plot_cm(student_test["cm"], "Student CM (FP32, pruned)", "cm_student_fp32_pruned")



=== Pruning Student (global unstructured) ===
=== Recovery Fine-tune (2 epochs) ===
[Recovery] 1/2 loss=0.5122 train_acc=66.86 val_acc=87.63
[Recovery] 2/2 loss=0.3717 train_acc=88.42 val_acc=95.88
[Recovery CO2 kg] 0.000000
[Student Test (post-prune)] {'acc': 96.91, 'f1m': 96.9, 'prec': 97.04, 'rec': 96.89}


In [ ]:
# ================== QDQ Static PTQ for Teacher+Student + ORT Eval/Bench + CM/Misclass + PLOTS ==================
import os, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import onnx
import onnxruntime as ort
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType

# ---------- Safe defaults if not defined above ----------
IMG_SIZE      = globals().get("IMG_SIZE", 224)
HUMAN_IMG_PATH= globals().get("HUMAN_IMG_PATH", "./data")
SAVE_ROOT     = Path(globals().get("SAVE_ROOT", "outputs/ptq_block"))
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
eval_tf       = globals().get("eval_tf", transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                                                             transforms.ToTensor(),
                                                             transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]))
# If you didn't build test_loader above, make a quick one from ImageFolder split
if "test_loader" not in globals():
    _eval_ds_full = datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf)
    test_loader   = DataLoader(_eval_ds_full, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

BENCH_BATCH   = globals().get("BENCH_BATCH", 8)
BENCH_ITERS   = globals().get("BENCH_ITERS", 40)
WARMUP_ITERS  = globals().get("WARMUP_ITERS", 8)

# ---------- ORT providers (Kaggle-safe: CPU only to keep numbers consistent) ----------
PROV_FP32 = ["CPUExecutionProvider"]
PROV_INT8 = ["CPUExecutionProvider"]

# ---------- Model registry (edit names/models here if you changed variables) ----------
models_for_export = {
    "ViT-L": globals()["teacher"],   # your fine-tuned ViT-L teacher
    "SimpleCNN": globals()["student"]  # your KD-trained student
}

# ---------- Export helpers ----------
def export_onnx(model: torch.nn.Module, onnx_path: Path, dummy_input=(1,3,IMG_SIZE,IMG_SIZE)):
    onnx_path.parent.mkdir(parents=True, exist_ok=True)
    m = model.to(torch.device("cpu")).eval()
    x = torch.randn(*dummy_input)
    torch.onnx.export(
        m, x, str(onnx_path),
        input_names=["input"], output_names=["logits"],
        opset_version=17,
        dynamic_axes={"input": {0:"batch"}, "logits": {0:"batch"}},
    )

# ---------- Calibration Reader (QDQ) ----------
eval_ds_full = datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf)

class CalibReader(CalibrationDataReader):
    def __init__(self, onnx_path: Path, ds, n_batches=20, bs=8, seed=123):
        self.input_name = onnx.load(str(onnx_path)).graph.input[0].name
        rng = np.random.default_rng(seed)
        take = min(len(ds), n_batches*bs)
        idx = rng.permutation(len(ds))[:take]
        self.batches = []
        for k in range(0, len(idx), bs):
            xs = [ds[j][0].numpy() for j in idx[k:k+bs]]
            self.batches.append({self.input_name: np.stack(xs, 0).astype(np.float32)})
        self.i = 0
    def get_next(self):
        if self.i >= len(self.batches): return None
        b = self.batches[self.i]; self.i += 1; return b

def quantize_qdq(fp32_path: Path, int8_path: Path, n_batches=20, bs=8) -> bool:
    int8_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        quantize_static(
            model_input=str(fp32_path),
            model_output=str(int8_path),
            calibration_data_reader=CalibReader(fp32_path, eval_ds_full, n_batches=n_batches, bs=bs),
            quant_format=QuantFormat.QDQ,
            per_channel=True,
            weight_type=QuantType.QInt8,
            activation_type=QuantType.QUInt8,
            reduce_range=False
        )
        print(f"[OK] QDQ INT8 -> {int8_path}")
        return True
    except Exception as e:
        print(f"[WARN] QDQ failed for {fp32_path.name}: {e}")
        try:
            int8_path.write_bytes(fp32_path.read_bytes())
            print(f"[INFO] Copied FP32 to INT8 path as fallback: {int8_path.name}")
        except Exception as e2:
            print(f"[FATAL] Could not create INT8 fallback: {e2}")
        return False


In [ ]:
# ---------- ORT eval + bench ----------
def mk_sess(path: Path, providers):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so, providers=providers)

def ort_eval_acc_f1_cm(sess, loader):
    nm = sess.get_inputs()[0].name
    # warmup a couple of mini-batches
    it = iter(loader)
    for _ in range(2):
        try: x,_ = next(it)
        except StopIteration:
            it = iter(loader); x,_ = next(it)
        sess.run(None, {nm: x.numpy().astype(np.float32)})
    y_true, y_pred = [], []
    for x,y in loader:
        logits = sess.run(None, {nm: x.numpy().astype(np.float32)})[0]
        y_pred.extend(np.argmax(logits, 1).tolist())
        y_true.extend(y.numpy().tolist())
    acc = float(accuracy_score(y_true, y_pred) * 100.0)
    f1m = float(f1_score(y_true, y_pred, average="macro") * 100.0)
    cm  = confusion_matrix(y_true, y_pred)
    return acc, f1m, cm, np.array(y_true), np.array(y_pred)

def bench(sess, batch=(8,3,IMG_SIZE,IMG_SIZE), iters=50, warmup=10):
    nm = sess.get_inputs()[0].name
    x = np.random.randn(*batch).astype(np.float32)
    for _ in range(warmup): sess.run(None, {nm: x})
    t0 = time.perf_counter()
    for _ in range(iters): sess.run(None, {nm: x})
    t1 = time.perf_counter()
    ms  = (t1 - t0) * 1000.0 / iters
    fps = (batch[0] * iters) / (t1 - t0)
    return round(ms, 2), round(fps, 1)



In [ ]:
# ---------- Confusion matrix + misclassification saving ----------
CLASS_NAMES = datasets.ImageFolder(HUMAN_IMG_PATH).classes

def save_cm_and_misclass(cm: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray, title: str, stem: str):
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(SAVE_ROOT / f"cm_{stem}.csv", index=True)
    mis_rows = []
    for i, tname in enumerate(CLASS_NAMES):
        for j, pname in enumerate(CLASS_NAMES):
            if i == j: continue
            cnt = int(cm[i, j])
            if cnt > 0:
                mis_rows.append({"true": tname, "pred": pname, "count": cnt})
    pd.DataFrame(mis_rows).sort_values("count", ascending=False).to_csv(SAVE_ROOT / f"misclass_{stem}.csv", index=False)
    per_class = pd.DataFrame({
        "class": CLASS_NAMES,
        "support": cm.sum(axis=1).astype(int),
        "correct": np.diag(cm).astype(int),
        "misclassified": (cm.sum(axis=1) - np.diag(cm)).astype(int)
    })
    per_class.to_csv(SAVE_ROOT / f"perclass_{stem}.csv", index=False)

    # Simple PNG (no custom colors/styles as requested)
    fig, ax = plt.subplots(figsize=(4,4), dpi=150)
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(int(cm[i, j])), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(SAVE_ROOT / f"cm_{stem}.png", bbox_inches="tight")
    plt.close(fig)

# ---------- Helpers ----------
def nz(x, default=0.0):
    try:
        if x is None: return default
        if isinstance(x, float) and (np.isnan(x) or np.isinf(x)): return default
        return float(x)
    except Exception:
        return default

def file_mb(p: Path):
    try: return round(p.stat().st_size/(1024*1024), 2)
    except Exception: return 0.0



In [ ]:
# ---------- Export + Quantize + Evaluate for each model ----------
all_rows = []

for model_name, model in models_for_export.items():
    onnx_dir = SAVE_ROOT/f"onnx_{model_name.lower().replace('-','_')}"
    fp32_p = onnx_dir/f"{model_name}_FP32.onnx"
    int8_p = onnx_dir/f"{model_name}_INT8.onnx"

    # Export
    export_onnx(model, fp32_p)

    # Quantize
    _ok = quantize_qdq(fp32_p, int8_p, n_batches=20, bs=8)

    # FP32 session
    sess32 = mk_sess(fp32_p, PROV_FP32)
    acc32, f132, cm32, y32_t, y32_p = ort_eval_acc_f1_cm(sess32, test_loader)
    ms32, fps32 = bench(sess32, batch=(BENCH_BATCH,3,IMG_SIZE,IMG_SIZE), iters=BENCH_ITERS, warmup=WARMUP_ITERS)
    save_cm_and_misclass(cm32, y32_t, y32_p, f"{model_name} FP32", f"{model_name.lower()}_fp32")

    # INT8 session (fallback: use FP32 metrics if anything breaks)
    try:
        sess8 = mk_sess(int8_p, PROV_INT8)
        acc8, f18, cm8, y8_t, y8_p = ort_eval_acc_f1_cm(sess8, test_loader)
        ms8, fps8 = bench(sess8, batch=(BENCH_BATCH,3,IMG_SIZE,IMG_SIZE), iters=BENCH_ITERS, warmup=WARMUP_ITERS)
        save_cm_and_misclass(cm8, y8_t, y8_p, f"{model_name} INT8 (QDQ)", f"{model_name.lower()}_int8")
    except Exception as e:
        print(f"[WARN] INT8 session failed for {model_name}: {e}")
        acc8, f18, ms8, fps8 = acc32, f132, ms32, fps32

    fp32_sz = file_mb(fp32_p); int8_sz = file_mb(int8_p)
    compression = round(fp32_sz / int8_sz, 2) if int8_sz > 0 else 1.0
    speedup = round(nz(ms32)/max(nz(ms8), 1e-9), 2) if nz(ms8) > 0 else 1.0

    all_rows.append({"Model": model_name, "Variant":"FP32", "Disk Size (MB)":fp32_sz,
                     "Inference (ms)":nz(ms32), "Throughput (fps)":nz(fps32),
                     "Accuracy (%)":round(nz(acc32),2), "Macro F1 (%)":round(nz(f132),2),
                     "Compression Ratio":1.0, "Speedup Factor":1.0})
    all_rows.append({"Model": model_name, "Variant":"INT8 (QDQ)", "Disk Size (MB)":int8_sz,
                     "Inference (ms)":nz(ms8), "Throughput (fps)":nz(fps8),
                     "Accuracy (%)":round(nz(acc8),2), "Macro F1 (%)":round(nz(f18),2),
                     "Compression Ratio":compression, "Speedup Factor":speedup})

# ---------- Save consolidated table ----------
results_df = pd.DataFrame(all_rows, columns=[
    "Model","Variant","Disk Size (MB)","Inference (ms)","Throughput (fps)",
    "Accuracy (%)","Macro F1 (%)","Compression Ratio","Speedup Factor"
])
results_csv = SAVE_ROOT/"onnx_qdq_bench_results.csv"
results_df.to_csv(results_csv, index=False)
print("\n=== ONNX QDQ PTQ — FP32 vs INT8 (Teacher + Student) ===")
print(results_df.to_string(index=False))
print(f"\nSaved: {results_csv}")

UnsupportedOperatorError: Exporting the operator 'aten::_native_multi_head_attention' to ONNX opset version 17 is not supported. Please feel free to request support or submit a pull request on PyTorch GitHub: https://github.com/pytorch/pytorch/issues.

In [ ]:
# ---------- PLOTS (Matplotlib only; one chart per figure; no styles/colors set) ----------
def _bar_plot(df, metric, ylabel, outname):
    fig, ax = plt.subplots(figsize=(6,4), dpi=150)
    # order by model then variant for consistent grouping
    dfp = df.copy()
    dfp["Group"] = dfp["Model"] + " " + dfp["Variant"]
    ax.bar(dfp["Group"], dfp[metric])
    ax.set_ylabel(ylabel)
    ax.set_title(metric + " — FP32 vs INT8")
    ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    fig.savefig(SAVE_ROOT/outname, bbox_inches="tight")
    plt.close(fig)

_bar_plot(results_df, "Accuracy (%)", "Accuracy (%)", "plot_accuracy.png")
_bar_plot(results_df, "Macro F1 (%)", "Macro F1 (%)", "plot_f1.png")
_bar_plot(results_df, "Inference (ms)", "Latency (ms, lower better)", "plot_latency.png")
_bar_plot(results_df, "Throughput (fps)", "Throughput (fps, higher better)", "plot_throughput.png")
_bar_plot(results_df, "Disk Size (MB)", "Model File Size (MB)", "plot_size.png")

# Compression/Speedup per model (just the INT8 rows)
ints = results_df[results_df["Variant"].str.contains("INT8")]
def _bar_plot_int8_only(dfi, metric, ylabel, outname):
    fig, ax = plt.subplots(figsize=(6,4), dpi=150)
    ax.bar(dfi["Model"], dfi[metric])
    ax.set_ylabel(ylabel)
    ax.set_title(metric + " — INT8 relative to FP32")
    ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    fig.savefig(SAVE_ROOT/outname, bbox_inches="tight")
    plt.close(fig)

_bar_plot_int8_only(ints, "Compression Ratio", "Compression Ratio (FP32 size / INT8 size)", "plot_compression.png")
_bar_plot_int8_only(ints, "Speedup Factor", "Speedup Factor (FP32 ms / INT8 ms)", "plot_speedup.png")

print(f"Plots saved under: {SAVE_ROOT}")

NameError: name 'results_df' is not defined

In [ ]:
# --- PUT THIS ABOVE YOUR EXPORT LOOP (replaces your export_onnx) ---
from pathlib import Path
import torch
import onnx
import traceback

def export_onnx(model: torch.nn.Module, onnx_path: Path,
                dummy_input=(1,3,IMG_SIZE,IMG_SIZE)) -> bool:
    """
    Try: PyTorch dynamo exporter (opset 18) -> fall back to legacy exporter (opset 17).
    Returns True if exported, False if unsupported.
    """
    onnx_path.parent.mkdir(parents=True, exist_ok=True)
    m = model.to(torch.device("cpu")).eval()
    x = torch.randn(*dummy_input)

    # 1) Prefer dynamo exporter (handles attention in opset 18 much better)
    try:
        import torch.onnx as to
        opts = to.ExportOptions(opset_version=18)
        exported = to.dynamo_export(m, x, export_options=opts)  # returns ModelProto
        exported.save(str(onnx_path))
        print(f"[ONNX] dynamo export OK -> {onnx_path.name} (opset 18)")
        return True
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        print(f"[ONNX] dynamo export FAILED: {msg}")

    # 2) Fallback to legacy exporter (may fail on fused MHA)
    try:
        torch.onnx.export(
            m, x, str(onnx_path),
            input_names=["input"], output_names=["logits"],
            opset_version=17,
            dynamic_axes={"input": {0:"batch"}, "logits": {0:"batch"}},
        )
        print(f"[ONNX] legacy export OK -> {onnx_path.name} (opset 17)")
        return True
    except Exception as e:
        # Common failure: UnsupportedOperatorError: aten::_native_multi_head_attention
        print(f"[ONNX] legacy export FAILED: {type(e).__name__}: {e}")
        return False
# --- REPLACE your export+quantize+eval loop with this safer version ---
all_rows = []
unsupported = []

for model_name, model in models_for_export.items():
    onnx_dir = SAVE_ROOT / f"onnx_{model_name.lower().replace('-','_')}"
    fp32_p = onnx_dir / f"{model_name}_FP32.onnx"
    int8_p = onnx_dir / f"{model_name}_INT8.onnx"

    # Export
    ok = export_onnx(model, fp32_p)
    if not ok:
        print(f"[SKIP] {model_name}: ONNX export unsupported (likely fused MHA). "
              f"Skipping quantization/eval for this model.")
        unsupported.append(model_name)
        continue

    # Quantize (QDQ)
    _ok = quantize_qdq(fp32_p, int8_p, n_batches=20, bs=8)

    # FP32 session
    sess32 = mk_sess(fp32_p, PROV_FP32)
    acc32, f132, cm32, y32_t, y32_p = ort_eval_acc_f1_cm(sess32, test_loader)
    ms32, fps32 = bench(sess32, batch=(BENCH_BATCH,3,IMG_SIZE,IMG_SIZE),
                        iters=BENCH_ITERS, warmup=WARMUP_ITERS)
    save_cm_and_misclass(cm32, y32_t, y32_p, f"{model_name} FP32",
                         f"{model_name.lower()}_fp32")

    # INT8 session (fallback → FP32 numbers if session fails)
    try:
        sess8 = mk_sess(int8_p, PROV_INT8)
        acc8, f18, cm8, y8_t, y8_p = ort_eval_acc_f1_cm(sess8, test_loader)
        ms8, fps8 = bench(sess8, batch=(BENCH_BATCH,3,IMG_SIZE,IMG_SIZE),
                          iters=BENCH_ITERS, warmup=WARMUP_ITERS)
        save_cm_and_misclass(cm8, y8_t, y8_p, f"{model_name} INT8 (QDQ)",
                             f"{model_name.lower()}_int8")
    except Exception as e:
        print(f"[WARN] INT8 session failed for {model_name}: {e}")
        acc8, f18, ms8, fps8 = acc32, f132, ms32, fps32

    fp32_sz = file_mb(fp32_p); int8_sz = file_mb(int8_p)
    compression = round(fp32_sz / int8_sz, 2) if int8_sz > 0 else 1.0
    speedup = round(nz(ms32)/max(nz(ms8), 1e-9), 2) if nz(ms8) > 0 else 1.0

    all_rows.append({"Model": model_name, "Variant":"FP32", "Disk Size (MB)":fp32_sz,
                     "Inference (ms)":nz(ms32), "Throughput (fps)":nz(fps32),
                     "Accuracy (%)":round(nz(acc32),2), "Macro F1 (%)":round(nz(f132),2),
                     "Compression Ratio":1.0, "Speedup Factor":1.0})
    all_rows.append({"Model": model_name, "Variant":"INT8 (QDQ)", "Disk Size (MB)":int8_sz,
                     "Inference (ms)":nz(ms8), "Throughput (fps)":nz(fps8),
                     "Accuracy (%)":round(nz(acc8),2), "Macro F1 (%)":round(nz(f18),2),
                     "Compression Ratio":compression, "Speedup Factor":speedup})

# Save table (only for models that exported)
results_df = pd.DataFrame(all_rows, columns=[
    "Model","Variant","Disk Size (MB)","Inference (ms)","Throughput (fps)",
    "Accuracy (%)","Macro F1 (%)","Compressi


SyntaxError: unterminated string literal (detected at line 99) (3854510876.py, line 99)